---
pinned_commit: d38a67e174f03a18a3817b82ff85fa80a287f9d0
---

# Stage 1 Control Feature Visualization V2

This notebook is a runnable audit harness for `train/stage1_oracle/features/control_v2.py`.
It covers review phases 2-6:

- Phase 2: synthetic pattern probes and edge/crop stability checks
- Phase 3: perturbation tests
- Phase 4: dataset distribution, saturation, confidence, and correlation audit
- Phase 5: top-section and disagreement outlier mining with local plots
- Phase 6: 4s/8s section embeddings and clustering inspection

The long dataset passes are disabled by default. The notebook recomputes V2 features from `.osu`
files because this repository currently has cached V1 control feature artifacts, not V2 parquets.


In [1]:
from __future__ import annotations

from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Sequence
import json
import math
import sys
import warnings

import matplotlib.pyplot as plt

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "requirements.txt").exists() and (candidate / "train").exists():
            return candidate
    raise FileNotFoundError("could not find repo root from current working directory")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from train.stage1_oracle.features.control import (  # noqa: E402
    HitObject,
    default_beat_length_at,
    mania_hit_objects_to_control_hits,
    red_timing_points_to_beat_length_fn,
)
from train.stage1_oracle.features.control_v2 import (  # noqa: E402
    CONFIDENCE_FEATURE_NAMES,
    DEBUG_ARRAY_NAMES,
    FEATURE_NAMES,
    MODEL_FEATURE_NAMES,
    VALUE_FEATURE_NAMES,
    FeatureConfigV2,
    extract_control_features,
)
from train.stage1_oracle.osu.hitobjects import parse_mania_hit_objects  # noqa: E402
from train.stage1_oracle.osu.timing import require_red_timing_points  # noqa: E402

DATASET_ROOT = REPO_ROOT / "mania-dataset"
INDEX_PATH = REPO_ROOT / "train/artifacts/indexes/beatmap_index_4k_no_timing_anomalies_2to6.parquet"
AUDIT_DIR = REPO_ROOT / "train/artifacts/features/control_v2_audit"
TIMESERIES_PATH = REPO_ROOT / "train/artifacts/features/control_v2_timeseries_4k_no_timing_anomalies_2to6.parquet"
SUMMARY_PATH = REPO_ROOT / "train/artifacts/features/control_v2_map_summary_4k_no_timing_anomalies_2to6.parquet"
METADATA_PATH = REPO_ROOT / "train/artifacts/features/control_v2_artifact_metadata_4k_no_timing_anomalies_2to6.json"
SECTION_AUDIT_PATH = AUDIT_DIR / "control_v2_section_audit_8s_stride4.parquet"
OUTLIER_DIR = AUDIT_DIR / "outliers"

CFG = FeatureConfigV2(grid_step=0.10)

VALUE_FEATURES = list(VALUE_FEATURE_NAMES)
CONFIDENCE_FEATURES = list(CONFIDENCE_FEATURE_NAMES)
MODEL_CHANNELS = list(MODEL_FEATURE_NAMES)
EXTRACTOR_OUTPUT_FEATURES = list(MODEL_FEATURE_NAMES)

if FEATURE_NAMES != MODEL_FEATURE_NAMES:
    raise RuntimeError("control_v2 FEATURE_NAMES and MODEL_FEATURE_NAMES disagree")

_missing_model_features = sorted(set(MODEL_CHANNELS) - set(EXTRACTOR_OUTPUT_FEATURES))
if _missing_model_features:
    raise RuntimeError(f"control_v2 MODEL_FEATURE_NAMES missing channels: {_missing_model_features}")

BOUNDED_SATURATION_FEATURES = [
    "chord_ratio",
    "hand_imbalance_abs",
    "repeat_exact",
    "repeat_shift",
    "repeat_motion",
    "repeat_rhythm",
]
SATURATION_THRESHOLDS = {
    "chord_ratio": (0.01, 0.95),
    "hand_imbalance_abs": (0.01, 0.95),
    "repeat_exact": (0.01, 0.95),
    "repeat_shift": (0.01, 0.95),
    "repeat_motion": (0.01, 0.95),
    "repeat_rhythm": (0.01, 0.95),
}
TAIL_AUDIT_FEATURES = [feature for feature in VALUE_FEATURES if feature not in SATURATION_THRESHOLDS]

FEATURE_CONFIDENCE_MAP = {
    "density_level": "density_confidence",
    "density_burst": "density_confidence",
    "ln_change_rate": "ln_change_confidence",
    "chord_ratio": "chord_confidence",
    "jack_excess": "jack_confidence",
    "jack_streak_exposure": "jack_streak_confidence",
    "hand_balance_signed": "hand_confidence",
    "hand_imbalance_abs": "hand_confidence",
    "repeat_exact": "repeat_confidence",
    "repeat_shift": "repeat_confidence",
    "repeat_motion": "repeat_confidence",
    "repeat_rhythm": "repeat_confidence",
}
PEAK_DEBUG_COLUMNS = {
    "density_level": {"raw": "density_raw_med", "n_eff": "density_n_eff_med"},
    "density_burst": {"raw": "density_raw_short", "n_eff": "density_n_eff_short"},
    "ln_change_rate": {"raw": "ln_change_raw", "n_eff": "ln_change_n_eff"},
    "chord_ratio": {"numerator": "chord_num", "denominator": "chord_den", "n_eff": "chord_n_eff", "raw": "chord_ratio_raw"},
    "jack_excess": {"numerator": "jack_observed", "denominator": "jack_expected_null", "n_eff": "jack_n_eff", "raw": "jack_excess_raw"},
    "jack_streak_exposure": {"raw": "jack_streak_raw", "n_eff": "jack_streak_n_eff", "max_streak": "jack_streak_max"},
    "hand_balance_signed": {"left_load": "hand_left_load", "right_load": "hand_right_load", "n_eff": "hand_n_eff", "raw": "hand_balance_raw"},
    "hand_imbalance_abs": {"left_load": "hand_left_load", "right_load": "hand_right_load", "n_eff": "hand_n_eff", "raw": "hand_balance_raw"},
    "repeat_exact": {"n_eff": "repeat_exact_n_eff", "top1_freq": "repeat_exact_top1_freq", "pattern_variety": "repeat_exact_pattern_variety"},
    "repeat_shift": {"n_eff": "repeat_shift_n_eff", "top1_freq": "repeat_shift_top1_freq", "pattern_variety": "repeat_shift_pattern_variety"},
    "repeat_motion": {"n_eff": "repeat_motion_n_eff", "top1_freq": "repeat_motion_top1_freq", "pattern_variety": "repeat_motion_pattern_variety"},
    "repeat_rhythm": {"n_eff": "repeat_rhythm_n_eff", "top1_freq": "repeat_rhythm_top1_freq", "pattern_variety": "repeat_rhythm_pattern_variety"},
}

DIAGNOSTIC_PLOT_FEATURES = [
    "density_level",
    "density_burst",
    "hold_occupancy",
    "ln_change_rate",
    "chord_ratio",
    "jack_excess",
    "jack_streak_exposure",
    "hand_balance_signed",
    "hand_imbalance_abs",
    "repeat_exact",
    "repeat_shift",
    "repeat_motion",
    "repeat_rhythm",
    "chord_confidence",
    "jack_confidence",
    "hand_confidence",
    "repeat_confidence",
    "control_confidence",
    "valid_control_mask",
]

DEBUG_ARRAY_COLUMNS = list(DEBUG_ARRAY_NAMES)

REPEAT_TOKEN_COLUMNS = [
    "repeat_exact_top_token",
    "repeat_shift_top_token",
    "repeat_motion_top_token",
    "repeat_rhythm_top_token",
]

print("repo root:", REPO_ROOT)
print("index:", INDEX_PATH)
print("audit dir:", AUDIT_DIR)
print("timeseries parquet:", TIMESERIES_PATH)
print("summary parquet:", SUMMARY_PATH)
print("artifact metadata:", METADATA_PATH)
print("V2 extractor outputs:", len(EXTRACTOR_OUTPUT_FEATURES), EXTRACTOR_OUTPUT_FEATURES)
print("V2 model channels:", len(MODEL_CHANNELS), MODEL_CHANNELS)


repo root: /Users/l/projects/Mapperatorinator
index: /Users/l/projects/Mapperatorinator/train/artifacts/indexes/beatmap_index_4k_no_timing_anomalies_2to6.parquet
audit dir: /Users/l/projects/Mapperatorinator/train/artifacts/features/control_v2_audit
timeseries parquet: /Users/l/projects/Mapperatorinator/train/artifacts/features/control_v2_timeseries_4k_no_timing_anomalies_2to6.parquet
summary parquet: /Users/l/projects/Mapperatorinator/train/artifacts/features/control_v2_map_summary_4k_no_timing_anomalies_2to6.parquet
artifact metadata: /Users/l/projects/Mapperatorinator/train/artifacts/features/control_v2_artifact_metadata_4k_no_timing_anomalies_2to6.json
V2 extractor outputs: 21 ['density_level', 'density_burst', 'hold_occupancy', 'ln_change_rate', 'chord_ratio', 'jack_excess', 'jack_streak_exposure', 'hand_balance_signed', 'hand_imbalance_abs', 'repeat_exact', 'repeat_shift', 'repeat_motion', 'repeat_rhythm', 'density_confidence', 'ln_change_confidence', 'chord_confidence', 'jack_co

In [2]:
if not INDEX_PATH.exists():
    raise FileNotFoundError(INDEX_PATH)
if not DATASET_ROOT.exists():
    raise FileNotFoundError(DATASET_ROOT)

metadata = pd.read_parquet(INDEX_PATH).reset_index(names="filtered_index")
metadata["difficulty"] = pd.to_numeric(metadata["difficulty"], errors="coerce")
metadata["beatmap_id"] = pd.to_numeric(metadata["beatmap_id"], errors="coerce")

assert "4k_no_timing_anomalies_2to6" in INDEX_PATH.name, f"unexpected phase index: {INDEX_PATH.name}"
assert metadata["difficulty"].notna().all(), "2-6★ index contains NaN difficulty"
assert metadata["difficulty"].between(2.0, 6.0, inclusive="both").all(), "index is not fully within 2-6★"
assert metadata["beatmap_path"].notna().all(), "index contains missing beatmap_path"
assert metadata["shard"].notna().all(), "index contains missing shard"
if "key_count" in metadata.columns:
    assert pd.to_numeric(metadata["key_count"], errors="coerce").eq(4).all(), "index contains non-4K maps"
if "mode" in metadata.columns:
    assert pd.to_numeric(metadata["mode"], errors="coerce").eq(3).all(), "index contains non-mania maps"


def search_maps(
    text: str | None = None,
    *,
    difficulty_min: float | None = None,
    difficulty_max: float | None = None,
    limit: int = 25,
    sort_by: str = "difficulty",
) -> pd.DataFrame:
    result = metadata.copy()
    if text:
        needle = text.casefold()
        haystack = (
            result["artist"].fillna("").astype(str)
            + " "
            + result["title"].fillna("").astype(str)
            + " "
            + result["version"].fillna("").astype(str)
            + " "
            + result["creator"].fillna("").astype(str)
        ).str.casefold()
        result = result.loc[haystack.str.contains(needle, regex=False)]
    if difficulty_min is not None:
        result = result.loc[result["difficulty"] >= difficulty_min]
    if difficulty_max is not None:
        result = result.loc[result["difficulty"] <= difficulty_max]
    if sort_by not in result.columns:
        raise ValueError(f"unknown sort column: {sort_by}")
    columns = [
        "filtered_index",
        "beatmap_id",
        "difficulty",
        "artist",
        "title",
        "version",
        "creator",
        "shard",
        "beatmap_path",
    ]
    return result.sort_values([sort_by, "filtered_index"])[columns].head(limit).reset_index(drop=True)


def resolve_beatmap(
    *,
    filtered_index: int | None = None,
    beatmap_id: int | None = None,
    text: str | None = None,
    rank: int = 0,
) -> pd.Series:
    selectors = [filtered_index is not None, beatmap_id is not None, text is not None]
    if not any(selectors):
        raise ValueError("provide filtered_index, beatmap_id, or text")

    mask = pd.Series(True, index=metadata.index)
    if filtered_index is not None:
        mask &= metadata["filtered_index"].eq(int(filtered_index))
    if beatmap_id is not None:
        mask &= metadata["beatmap_id"].eq(int(beatmap_id))
    if text is not None:
        needle = text.casefold()
        haystack = (
            metadata["artist"].fillna("").astype(str)
            + " "
            + metadata["title"].fillna("").astype(str)
            + " "
            + metadata["version"].fillna("").astype(str)
            + " "
            + metadata["creator"].fillna("").astype(str)
        ).str.casefold()
        mask &= haystack.str.contains(needle, regex=False)

    matches = metadata.loc[mask].sort_values(["difficulty", "filtered_index"])
    if matches.empty:
        raise LookupError("no beatmap matched the provided selector")
    if rank < 0 or rank >= len(matches):
        raise IndexError(f"rank {rank} outside {len(matches)} matches")
    if len(matches) > 1:
        print(f"matched {len(matches)} maps; using rank={rank}. Use search_maps(...) to inspect candidates.")
    return matches.iloc[rank]


def beatmap_path_for(row: pd.Series) -> Path:
    return DATASET_ROOT / str(row["shard"]) / str(row["beatmap_path"])


print("maps:", len(metadata))
print("difficulty range:", float(metadata["difficulty"].min()), "to", float(metadata["difficulty"].max()))
display(search_maps(difficulty_min=4.0, difficulty_max=4.1, limit=5))


maps: 10977
difficulty range: 2.0 to 6.0


,filtered_index,beatmap_id,difficulty,artist,title,version,creator,shard,beatmap_path
0,74,2131277,4.0,Fractal Dreamers,Celestial Horizon,Heavenly,Voxa,0,1018454/Fractal Dreamers - Celestial Horizon (...
1,93,2261935,4.0,Roselia,BRAVE JEWEL (TV Size),Insane,KonGuuuu,0,1024827/Roselia - BRAVE JEWEL (TV Size) (KonGu...
2,302,3692919,4.0,goreshit,najimi breakers,hard,gogozzzx,0,1079652/goreshit - najimi breakers (gogozzzx) ...
3,466,2321696,4.0,CHUBAY,Itadaki,Insane,MarioUniverseZ,0,1111192/CHUBAY - Itadaki (MarioUniverseZ) [Ins...
4,1118,2613373,4.0,ARForest,Regret,Insane,FAMoss,0,1254196/ARForest - Regret (FAMoss) [Insane].osu


In [3]:
for path in (TIMESERIES_PATH, SUMMARY_PATH, METADATA_PATH):
    if not path.exists():
        raise FileNotFoundError(path)

artifact_metadata = json.loads(METADATA_PATH.read_text())
summary_df = pd.read_parquet(SUMMARY_PATH)

expected_timeseries = TIMESERIES_PATH.relative_to(REPO_ROOT).as_posix()
expected_summary = SUMMARY_PATH.relative_to(REPO_ROOT).as_posix()
assert artifact_metadata.get("timeseries_path") == expected_timeseries, artifact_metadata.get("timeseries_path")
assert artifact_metadata.get("summary_path") == expected_summary, artifact_metadata.get("summary_path")
assert int(artifact_metadata.get("map_count", -1)) == len(metadata) == len(summary_df)
assert int(artifact_metadata.get("error_count", -1)) == 0
assert summary_df["filtered_index"].nunique() == len(metadata)
assert set(summary_df["filtered_index"].astype(int)) == set(metadata["filtered_index"].astype(int))

print("artifact map_count:", artifact_metadata["map_count"])
print("artifact timeseries_rows:", artifact_metadata["timeseries_rows"])
print("artifact error_count:", artifact_metadata["error_count"])
print("summary rows:", len(summary_df))
display(summary_df[["filtered_index", "beatmap_id", "difficulty", "finite", "ranges_ok", "error_type", "error"]].head())


artifact map_count: 10977
artifact timeseries_rows: 16858417
artifact error_count: 0
summary rows: 10977


,filtered_index,beatmap_id,difficulty,finite,ranges_ok,error_type,error
0,0,2092272,4.49,True,True,,
1,1,2093729,4.80,True,True,,
2,2,2096823,3.09,True,True,,
3,3,2173978,2.00,True,True,,
4,4,2156259,3.39,True,True,,


In [4]:
@dataclass(frozen=True)
class MapInputs:
    row: pd.Series | None
    beatmap_path: Path | None
    hits: list[HitObject]
    beat_length_at: Callable[[float], float]
    bpm_median: float
    map_duration_s: float


def load_map_inputs(row_or_filtered_index: pd.Series | int) -> MapInputs:
    row = resolve_beatmap(filtered_index=int(row_or_filtered_index)) if isinstance(row_or_filtered_index, int) else row_or_filtered_index
    beatmap_path = beatmap_path_for(row)
    hitobjects = parse_mania_hit_objects(beatmap_path, expected_key_count=4)
    timing_points = require_red_timing_points(beatmap_path)
    hits = mania_hit_objects_to_control_hits(hitobjects)
    beat_length_at = red_timing_points_to_beat_length_fn(timing_points)
    map_duration_s = max((hit.end if hit.end is not None else hit.start for hit in hits), default=0.0)
    bpms = [60000.0 / point.beat_length_ms for point in timing_points if point.beat_length_ms > 0]
    bpm_median = float(np.median(bpms)) if bpms else math.nan
    return MapInputs(
        row=row,
        beatmap_path=beatmap_path,
        hits=hits,
        beat_length_at=beat_length_at,
        bpm_median=bpm_median,
        map_duration_s=float(map_duration_s),
    )


def debug_array_or_none(out: dict[str, Any], name: str) -> np.ndarray | None:
    value = out["debug"].get(name)
    if value is None:
        return None
    array = np.asarray(value)
    if array.shape != out["time"].shape:
        return None
    return array


def frame_from_control_output(out: dict[str, Any], include_debug: bool = True) -> pd.DataFrame:
    frame = pd.DataFrame({"time_s": out["time"]})
    model_features = out.get("model_feature_names", out.get("feature_names", MODEL_CHANNELS))
    for name in model_features:
        frame[name] = out["features"][name]
    if include_debug:
        for name in DEBUG_ARRAY_COLUMNS:
            array = debug_array_or_none(out, name)
            if array is not None:
                if name in frame.columns:
                    frame[f"{name}_debug_raw"] = array
                else:
                    frame[name] = array
    return frame


def extract_v2_from_hits(
    hits: Sequence[HitObject],
    *,
    beat_length_at: Callable[[float], float] = default_beat_length_at,
    cfg: FeatureConfigV2 = CFG,
    start_time: float | None = None,
    end_time: float | None = None,
    grid: np.ndarray | None = None,
    include_debug: bool = True,
) -> tuple[dict[str, Any], pd.DataFrame]:
    out = extract_control_features(
        hits,
        beat_length_at=beat_length_at,
        cfg=cfg,
        start_time=start_time,
        end_time=end_time,
        grid=grid,
        return_debug=include_debug,
    )
    return out, frame_from_control_output(out, include_debug=include_debug)


def extract_v2_for_row(
    row_or_filtered_index: pd.Series | int,
    *,
    cfg: FeatureConfigV2 = CFG,
    start_time: float | None = None,
    end_time: float | None = None,
    grid: np.ndarray | None = None,
    include_debug: bool = True,
) -> tuple[MapInputs, dict[str, Any], pd.DataFrame]:
    inputs = load_map_inputs(row_or_filtered_index)
    if end_time is None and grid is None:
        end_time = inputs.map_duration_s
    out, frame = extract_v2_from_hits(
        inputs.hits,
        beat_length_at=inputs.beat_length_at,
        cfg=cfg,
        start_time=start_time,
        end_time=end_time,
        grid=grid,
        include_debug=include_debug,
    )
    return inputs, out, frame


def valid_values(frame: pd.DataFrame, feature: str) -> np.ndarray:
    values = frame[feature].to_numpy(dtype=float)
    if "valid_control_mask" in frame.columns:
        mask = frame["valid_control_mask"].to_numpy(dtype=bool)
        values = values[mask]
    values = values[np.isfinite(values)]
    return values


def feature_summary_row(frame: pd.DataFrame, features: Sequence[str] = MODEL_CHANNELS) -> dict[str, float]:
    row: dict[str, float] = {}
    for feature in features:
        if feature not in frame.columns:
            continue
        values = valid_values(frame, feature)
        if len(values) == 0:
            row[f"{feature}_mean"] = 0.0
            row[f"{feature}_p95"] = 0.0
            row[f"{feature}_max"] = 0.0
            continue
        row[f"{feature}_mean"] = float(np.mean(values))
        row[f"{feature}_p95"] = float(np.percentile(values, 95))
        row[f"{feature}_max"] = float(np.max(values))
    return row


def print_feature_ranges(frame: pd.DataFrame, features: Sequence[str] = MODEL_CHANNELS) -> pd.DataFrame:
    rows = []
    for feature in features:
        values = valid_values(frame, feature)
        rows.append({
            "feature": feature,
            "min": float(np.min(values)) if len(values) else 0.0,
            "mean": float(np.mean(values)) if len(values) else 0.0,
            "p95": float(np.percentile(values, 95)) if len(values) else 0.0,
            "max": float(np.max(values)) if len(values) else 0.0,
        })
    return pd.DataFrame(rows)


In [5]:
def _section_objects(hits: Sequence[HitObject], start_s: float, end_s: float) -> list[HitObject]:
    selected = []
    for hit in hits:
        hit_end = hit.end if hit.end is not None else hit.start
        if hit.start <= end_s and hit_end >= start_s:
            selected.append(hit)
    return selected


def plot_local_section(
    inputs: MapInputs,
    out: dict[str, Any],
    frame: pd.DataFrame,
    *,
    start_s: float,
    window_s: float = 12.0,
    features: Sequence[str] = DIAGNOSTIC_PLOT_FEATURES,
    max_points: int = 2000,
    title: str | None = None,
):
    end_s = start_s + window_s
    available_features = [feature for feature in features if feature in frame.columns]
    local = frame.loc[(frame["time_s"] >= start_s) & (frame["time_s"] <= end_s), ["time_s", *available_features]].copy()
    if len(local) > max_points:
        step = max(1, math.ceil(len(local) / max_points))
        local = local.iloc[::step].reset_index(drop=True)

    fig, axes = plt.subplots(len(available_features) + 1, 1, sharex=True, figsize=(16, 1.45 * (len(available_features) + 1)))
    axes = np.atleast_1d(axes)
    object_ax = axes[0]
    object_ax.set_ylabel("col")
    object_ax.set_ylim(-0.5, 3.5)
    object_ax.set_yticks([0, 1, 2, 3])
    object_ax.grid(True, axis="x", alpha=0.2)
    for hit in _section_objects(inputs.hits, start_s, end_s):
        hit_end = hit.end if hit.end is not None else hit.start
        x0 = max(start_s, hit.start)
        x1 = min(end_s, max(hit_end, hit.start + 0.015))
        linewidth = 7.0 if hit_end > hit.start + 0.03 else 5.0
        object_ax.plot([x0, x1], [hit.col, hit.col], linewidth=linewidth, solid_capstyle="butt")

    for ax, feature in zip(axes[1:], available_features):
        ax.plot(local["time_s"], local[feature].astype(float), linewidth=1.2)
        ax.set_ylabel(feature)
        ax.grid(True, alpha=0.25)
        if feature == "hand_balance_signed":
            ax.axhline(0.0, color="black", linewidth=0.8, alpha=0.4)
    axes[-1].set_xlabel("time from audio start (s)")
    axes[-1].set_xlim(start_s, end_s)

    if title is None and inputs.row is not None:
        row = inputs.row
        title = (
            f"{row['artist']} - {row['title']} [{row['version']}] | "
            f"filtered_index={int(row['filtered_index'])}, stars={float(row['difficulty']):.2f}"
        )
    if title:
        fig.suptitle(title, y=1.0)
    fig.tight_layout()
    return fig


def repeat_top_tokens_over_timegrid(
    out: dict[str, Any],
    *,
    start_s: float,
    end_s: float,
    family: str,
    top_n: int = 10,
) -> pd.DataFrame:
    key = f"repeat_{family}_top_token"
    if key not in out["debug"]:
        raise KeyError(key)
    time = np.asarray(out["time"], dtype=float)
    tokens = out["debug"][key]
    mask = (time >= start_s) & (time <= end_s)
    counter = Counter(repr(token) for token in np.asarray(tokens, dtype=object)[mask] if token is not None)
    total = sum(counter.values())
    return pd.DataFrame(
        [
            {
                "token": token,
                "grid_rows": count,
                "share": count / total if total else 0.0,
                "token_source": "top_token_over_timegrid",
            }
            for token, count in counter.most_common(top_n)
        ]
    )


def save_local_section_plot(
    output_path: str | Path,
    inputs: MapInputs,
    out: dict[str, Any],
    frame: pd.DataFrame,
    *,
    start_s: float,
    window_s: float = 12.0,
    features: Sequence[str] = DIAGNOSTIC_PLOT_FEATURES,
) -> Path:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig = plot_local_section(inputs, out, frame, start_s=start_s, window_s=window_s, features=features)
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    return output_path


## Phase 2: Synthetic Pattern Probes

These fixtures are intentionally small and deterministic. They validate that V2 operators respond to
known control regimes before running dataset-scale audits. The expectation table should be treated as
a unit-test draft: failures are useful because they point to token granularity, confidence gating, or
null-baseline issues before the expensive full-corpus pass.


In [6]:
def _add_event(hits: list[HitObject], t: float, cols: Sequence[int], duration: float = 0.0) -> None:
    for col in cols:
        hits.append(HitObject(col=int(col), start=float(t), end=float(t + duration) if duration > 0 else None))


def taps_from_pattern(
    pattern: Sequence[Sequence[int]],
    *,
    interval_s: float,
    duration_s: float,
    start_s: float = 0.0,
) -> list[HitObject]:
    hits: list[HitObject] = []
    t = float(start_s)
    i = 0
    while t <= duration_s + 1e-9:
        _add_event(hits, t, pattern[i % len(pattern)])
        t += interval_s
        i += 1
    return hits


def fixture_rest(duration_s: float = 16.0) -> list[HitObject]:
    return []


def fixture_single_stream(duration_s: float = 16.0) -> list[HitObject]:
    return taps_from_pattern([(0,), (1,), (2,), (3,)], interval_s=0.25, duration_s=duration_s)


def fixture_random_stream(duration_s: float = 16.0, seed: int = 7) -> list[HitObject]:
    rng = np.random.default_rng(seed)
    hits: list[HitObject] = []
    t = 0.0
    while t <= duration_s + 1e-9:
        _add_event(hits, t, (int(rng.integers(0, 4)),))
        t += 0.125
    return hits


def fixture_chordstream(duration_s: float = 16.0) -> list[HitObject]:
    return taps_from_pattern([(0, 1), (2, 3), (0, 2), (1, 3)], interval_s=0.25, duration_s=duration_s)


def fixture_jumpstream(duration_s: float = 16.0) -> list[HitObject]:
    return taps_from_pattern([(0,), (1, 3), (2,), (0, 3), (1,), (0, 2)], interval_s=0.18, duration_s=duration_s)


def fixture_normal_jack(duration_s: float = 16.0) -> list[HitObject]:
    return taps_from_pattern([(0,)], interval_s=0.20, duration_s=duration_s)


def fixture_stress_jack(duration_s: float = 16.0) -> list[HitObject]:
    return taps_from_pattern([(0,)], interval_s=0.10, duration_s=duration_s)


def fixture_minijack(duration_s: float = 16.0) -> list[HitObject]:
    return taps_from_pattern([(0,), (0,), (1,), (1,), (2,), (2,), (3,), (3,)], interval_s=0.12, duration_s=duration_s)


def fixture_anchor(duration_s: float = 16.0) -> list[HitObject]:
    return taps_from_pattern([(0,), (0, 1), (0,), (0, 2), (0,), (0, 3)], interval_s=0.12, duration_s=duration_s)


def fixture_single_stair(duration_s: float = 16.0) -> list[HitObject]:
    return taps_from_pattern([(0,), (1,), (2,), (3,)], interval_s=0.30, duration_s=duration_s)


def fixture_double_stair(duration_s: float = 16.0) -> list[HitObject]:
    return taps_from_pattern([(0, 1), (1, 2), (2, 3)], interval_s=0.45, duration_s=duration_s)


def fixture_hand_alternation(duration_s: float = 16.0) -> list[HitObject]:
    return taps_from_pattern([(0,), (3,), (1,), (2,)], interval_s=0.14, duration_s=duration_s)


def fixture_normal_roll(duration_s: float = 16.0) -> list[HitObject]:
    return taps_from_pattern([(0,), (1,), (2,), (3,)], interval_s=0.16, duration_s=duration_s)


def fixture_stress_roll(duration_s: float = 16.0) -> list[HitObject]:
    return taps_from_pattern([(0,), (1,), (2,), (3,)], interval_s=0.08, duration_s=duration_s)


def fixture_ln_hold_section(duration_s: float = 16.0) -> list[HitObject]:
    return [
        HitObject(col=0, start=2.0, end=14.0),
        HitObject(col=2, start=3.0, end=13.0),
        HitObject(col=1, start=6.0, end=12.0),
    ]


def fixture_ln_release_heavy_section(duration_s: float = 16.0) -> list[HitObject]:
    return [HitObject(col=i % 4, start=i * 0.35, end=i * 0.35 + 0.18) for i in range(40)]


def fixture_mixed_ln_chord_section(duration_s: float = 16.0) -> list[HitObject]:
    hits = [HitObject(col=0, start=1.0, end=9.0), HitObject(col=3, start=4.0, end=12.0)]
    hits.extend(taps_from_pattern([(1, 2), (0, 3), (1, 3)], interval_s=0.30, duration_s=duration_s))
    return sorted(hits, key=lambda hit: hit.start)


def fixture_exact_loop_non_jack(duration_s: float = 16.0) -> list[HitObject]:
    return taps_from_pattern([(0,), (2,)], interval_s=0.50, duration_s=duration_s)


def fixture_shifted_loop_non_jack(duration_s: float = 16.0) -> list[HitObject]:
    return taps_from_pattern([(0, 1), (1, 2), (2, 3), (1, 2)], interval_s=0.50, duration_s=duration_s)


def fixture_motion_loop_non_shift(duration_s: float = 16.0) -> list[HitObject]:
    return taps_from_pattern([(0,), (2,), (0, 1), (2, 3), (1,), (3,), (0, 2), (1, 3)], interval_s=0.50, duration_s=duration_s)


def fixture_rhythm_repeat_random_columns(duration_s: float = 16.0, seed: int = 11) -> list[HitObject]:
    rng = np.random.default_rng(seed)
    hits: list[HitObject] = []
    t = 0.0
    gaps = [0.18, 0.36, 0.18, 0.36]
    i = 0
    while t <= duration_s + 1e-9:
        _add_event(hits, t, (int(rng.integers(0, 4)),))
        t += gaps[i % len(gaps)]
        i += 1
    return hits


def fixture_same_density_random_control(duration_s: float = 16.0, seed: int = 13) -> list[HitObject]:
    rng = np.random.default_rng(seed)
    hits: list[HitObject] = []
    t = 0.0
    while t <= duration_s + 1e-9:
        _add_event(hits, t, (int(rng.integers(0, 4)),))
        t += 0.25
    return hits


SYNTHETIC_FIXTURES: dict[str, Callable[[float], list[HitObject]]] = {
    "rest": fixture_rest,
    "single_stream": fixture_single_stream,
    "random_stream": fixture_random_stream,
    "chordstream": fixture_chordstream,
    "jumpstream": fixture_jumpstream,
    "normal_jack": fixture_normal_jack,
    "stress_jack": fixture_stress_jack,
    "minijack": fixture_minijack,
    "anchor": fixture_anchor,
    "single_stair": fixture_single_stair,
    "double_stair": fixture_double_stair,
    "hand_alternation": fixture_hand_alternation,
    "normal_roll": fixture_normal_roll,
    "stress_roll": fixture_stress_roll,
    "ln_hold_section": fixture_ln_hold_section,
    "ln_release_heavy_section": fixture_ln_release_heavy_section,
    "mixed_ln_chord_section": fixture_mixed_ln_chord_section,
    "exact_loop_non_jack": fixture_exact_loop_non_jack,
    "shifted_loop_non_jack": fixture_shifted_loop_non_jack,
    "motion_loop_non_shift": fixture_motion_loop_non_shift,
    "rhythm_repeat_random_columns": fixture_rhythm_repeat_random_columns,
    "same_density_random_control": fixture_same_density_random_control,
}


def synthetic_probe_outputs(
    *,
    duration_s: float = 16.0,
    cfg: FeatureConfigV2 = CFG,
) -> tuple[dict[str, tuple[list[HitObject], dict[str, Any], pd.DataFrame]], pd.DataFrame]:
    outputs: dict[str, tuple[list[HitObject], dict[str, Any], pd.DataFrame]] = {}
    rows = []
    for name, builder in SYNTHETIC_FIXTURES.items():
        hits = builder(duration_s)
        out, frame = extract_v2_from_hits(
            hits,
            cfg=cfg,
            start_time=0.0,
            end_time=duration_s,
            include_debug=True,
        )
        outputs[name] = (hits, out, frame)
        row = {"fixture": name, "objects": len(hits), **feature_summary_row(frame, EXTRACTOR_OUTPUT_FEATURES)}
        rows.append(row)
    return outputs, pd.DataFrame(rows)


@dataclass(frozen=True)
class SyntheticExpectation:
    fixture: str
    feature: str
    statistic: str
    op: str
    threshold: float
    reason: str


SYNTHETIC_EXPECTATIONS = [
    SyntheticExpectation("rest", "density_level", "max", "<=", 0.0, "rest should stay neutral"),
    SyntheticExpectation("rest", "repeat_exact", "max", "<=", 0.0, "no objects means no recurrence"),
    SyntheticExpectation("normal_jack", "jack_excess", "p95", ">=", 0.30, "ranked-normal same-column gaps should exceed null"),
    SyntheticExpectation("normal_jack", "repeat_exact", "p95", ">=", 0.50, "literal recurrence should be high on normal jack"),
    SyntheticExpectation("single_stair", "repeat_motion", "p95", ">=", 0.10, "single-column movement recurrence should fire"),
    SyntheticExpectation("single_stair", "jack_excess", "p95", "<=", 0.05, "single stair should not look like jack risk"),
    SyntheticExpectation("double_stair", "repeat_shift", "p95", ">=", 0.45, "shift-normalized chord-shape recurrence should fire"),
    SyntheticExpectation("double_stair", "jack_excess", "p95", "<=", 0.05, "double stair should not look like jack risk"),
    SyntheticExpectation("jumpstream", "density_level", "p95", ">=", 1.80, "single-chord jumpstream should be high-density"),
    SyntheticExpectation("jumpstream", "chord_ratio", "p95", ">=", 0.15, "jumpstream alternates singles and chords"),
    SyntheticExpectation("jumpstream", "repeat_motion", "p95", ">=", 0.10, "jumpstream should have movement grammar recurrence"),
    SyntheticExpectation("random_stream", "repeat_exact", "p95", "<=", 0.45, "random stream should not have strong exact recurrence"),
    SyntheticExpectation("random_stream", "jack_excess", "p95", "<=", 0.60, "random same-column collisions should be below true jack"),
    SyntheticExpectation("ln_hold_section", "hold_occupancy", "p95", ">=", 0.60, "long holds should occupy columns"),
    SyntheticExpectation("ln_release_heavy_section", "ln_change_rate", "p95", ">=", 1.00, "many LN transitions should raise change rate"),
    SyntheticExpectation("mixed_ln_chord_section", "hold_occupancy", "p95", ">=", 0.30, "mixed fixture should keep LN pressure"),
    SyntheticExpectation("mixed_ln_chord_section", "chord_ratio", "p95", ">=", 0.20, "mixed fixture should keep chord pressure"),
    SyntheticExpectation("exact_loop_non_jack", "repeat_exact", "p95", ">=", 0.30, "exact loop should trigger exact recurrence without short jacks"),
    SyntheticExpectation("exact_loop_non_jack", "jack_excess", "p95", "<=", 0.05, "slow exact loop should not become jack excess"),
    SyntheticExpectation("shifted_loop_non_jack", "repeat_shift", "p95", ">=", 0.30, "shifted loop should trigger shift recurrence"),
    SyntheticExpectation("shifted_loop_non_jack", "jack_excess", "p95", "<=", 0.05, "slow shifted loop should not become jack excess"),
    SyntheticExpectation("rhythm_repeat_random_columns", "repeat_rhythm", "p95", ">=", 0.30, "fixed rhythm should trigger rhythm recurrence even with random columns"),
    SyntheticExpectation("same_density_random_control", "density_level", "p95", ">=", 1.40, "same-density random control should preserve stream density"),
]


def _compare(value: float, op: str, threshold: float) -> bool:
    if op == ">=":
        return value >= threshold
    if op == "<=":
        return value <= threshold
    if op == ">":
        return value > threshold
    if op == "<":
        return value < threshold
    raise ValueError(f"unknown op: {op}")


def evaluate_synthetic_expectations(summary: pd.DataFrame) -> pd.DataFrame:
    by_fixture = summary.set_index("fixture")
    rows = []
    for item in SYNTHETIC_EXPECTATIONS:
        column = f"{item.feature}_{item.statistic}"
        value = float(by_fixture.loc[item.fixture, column])
        rows.append({
            "fixture": item.fixture,
            "feature": item.feature,
            "statistic": item.statistic,
            "value": value,
            "op": item.op,
            "threshold": item.threshold,
            "pass": _compare(value, item.op, item.threshold),
            "reason": item.reason,
        })
    return pd.DataFrame(rows)


def edge_default_report(hits: Sequence[HitObject], *, duration_s: float = 16.0, pad_s: float = 2.0) -> pd.DataFrame:
    out, frame = extract_v2_from_hits(
        hits,
        start_time=-pad_s,
        end_time=duration_s + pad_s,
        include_debug=True,
    )
    invalid = ~frame["valid_control_mask"].to_numpy(dtype=bool)
    rows = []
    for feature in EXTRACTOR_OUTPUT_FEATURES:
        values = frame[feature].to_numpy(dtype=float)
        rows.append({
            "feature": feature,
            "invalid_abs_max": float(np.max(np.abs(values[invalid]))) if invalid.any() else 0.0,
            "invalid_rows": int(invalid.sum()),
        })
    return pd.DataFrame(rows)


def crop_stability_report(
    hits: Sequence[HitObject],
    *,
    duration_s: float = 16.0,
    crop_s: float = 2.0,
    margin_s: float = 4.0,
    features: Sequence[str] = MODEL_CHANNELS,
) -> pd.DataFrame:
    compare_start = crop_s + margin_s
    compare_end = duration_s - crop_s - margin_s
    if compare_end < compare_start:
        raise ValueError(
            f"crop_s={crop_s} and margin_s={margin_s} leave no interior for duration_s={duration_s}"
        )
    grid = np.arange(compare_start, compare_end + 0.5 * CFG.grid_step, CFG.grid_step)
    original_out, original_frame = extract_v2_from_hits(hits, grid=grid, include_debug=True)
    cropped_hits = [
        HitObject(col=hit.col, start=hit.start, end=hit.end)
        for hit in hits
        if crop_s <= hit.start <= duration_s - crop_s
    ]
    cropped_out, cropped_frame = extract_v2_from_hits(cropped_hits, grid=grid, include_debug=True)
    if "valid_control_mask" in original_frame and "valid_control_mask" in cropped_frame:
        mask = original_frame["valid_control_mask"].to_numpy(dtype=bool) & cropped_frame["valid_control_mask"].to_numpy(dtype=bool)
    else:
        mask = np.ones(len(grid), dtype=bool)
    rows = []
    for feature in features:
        if feature not in original_frame or feature not in cropped_frame:
            continue
        original = original_frame.loc[mask, feature].to_numpy(dtype=float)
        cropped = cropped_frame.loc[mask, feature].to_numpy(dtype=float)
        rows.append({
            "feature": feature,
            "compare_start_s": float(compare_start),
            "compare_end_s": float(compare_end),
            "grid_rows": int(len(original)),
            "mean_abs_delta": float(np.mean(np.abs(original - cropped))) if len(original) else 0.0,
            "max_abs_delta": float(np.max(np.abs(original - cropped))) if len(original) else 0.0,
        })
    return pd.DataFrame(rows)


## Phase 3: Perturbation Tests

Each perturbation checks that the intended operator changes while unrelated operators remain mostly stable.
The tables below report deltas rather than hiding them behind a single pass/fail score; review the largest
unexpected deltas before promoting these into automated tests.


In [7]:
def mirror_columns(hits: Sequence[HitObject], key_count: int = 4) -> list[HitObject]:
    return [HitObject(col=key_count - 1 - hit.col, start=hit.start, end=hit.end) for hit in hits]


def time_stretch(hits: Sequence[HitObject], factor: float = 1.1) -> list[HitObject]:
    return [
        HitObject(
            col=hit.col,
            start=hit.start * factor,
            end=(hit.end * factor if hit.end is not None else None),
        )
        for hit in hits
    ]


def stretch_beat_length_fn(old_fn: Callable[[float], float], factor: float) -> Callable[[float], float]:
    return lambda t: old_fn(t / factor) * factor


def _group_hits_by_start(hits: Sequence[HitObject], eps: float = 0.002) -> list[list[HitObject]]:
    ordered = sorted(hits, key=lambda hit: hit.start)
    groups: list[list[HitObject]] = []
    current: list[HitObject] = []
    ref_t: float | None = None
    for hit in ordered:
        if ref_t is None or hit.start - ref_t > eps:
            if current:
                groups.append(current)
            current = [hit]
            ref_t = hit.start
        else:
            current.append(hit)
    if current:
        groups.append(current)
    return groups


def column_shuffle_preserving_chord_size(
    hits: Sequence[HitObject],
    *,
    key_count: int = 4,
    seed: int = 7,
) -> list[HitObject]:
    rng = np.random.default_rng(seed)
    shuffled: list[HitObject] = []
    for group in _group_hits_by_start(hits):
        cols = rng.choice(key_count, size=min(len(group), key_count), replace=False)
        for hit, col in zip(group, cols):
            shuffled.append(HitObject(col=int(col), start=hit.start, end=hit.end))
    return sorted(shuffled, key=lambda hit: hit.start)


def randomize_event_order_within_section(hits: Sequence[HitObject], *, seed: int = 7) -> list[HitObject]:
    rng = np.random.default_rng(seed)
    groups = _group_hits_by_start(hits)
    payloads = [[(hit.col, (hit.end - hit.start) if hit.end is not None else 0.0) for hit in group] for group in groups]
    rng.shuffle(payloads)
    randomized: list[HitObject] = []
    for group, payload in zip(groups, payloads):
        start = float(np.median([hit.start for hit in group]))
        for col, duration in payload:
            randomized.append(HitObject(col=int(col), start=start, end=start + duration if duration > 0 else None))
    return sorted(randomized, key=lambda hit: hit.start)


def force_same_column_repeats(hits: Sequence[HitObject], col: int = 0) -> list[HitObject]:
    forced: list[HitObject] = []
    for group in _group_hits_by_start(hits):
        if len(group) == 1:
            hit = group[0]
            forced.append(HitObject(col=col, start=hit.start, end=hit.end))
        else:
            for i, hit in enumerate(group):
                forced.append(HitObject(col=(col + i) % 4, start=hit.start, end=hit.end))
    return sorted(forced, key=lambda hit: hit.start)


def convert_singles_to_doubles(hits: Sequence[HitObject], *, key_count: int = 4, offset: int = 1) -> list[HitObject]:
    converted: list[HitObject] = []
    for group in _group_hits_by_start(hits):
        if len(group) != 1:
            converted.extend(HitObject(col=hit.col, start=hit.start, end=hit.end) for hit in group)
            continue
        hit = group[0]
        cols = [hit.col, (hit.col + offset) % key_count]
        for col in dict.fromkeys(cols):
            converted.append(HitObject(col=int(col), start=hit.start, end=hit.end))
    return sorted(converted, key=lambda hit: hit.start)


def remove_ln_tails(hits: Sequence[HitObject]) -> list[HitObject]:
    return [HitObject(col=hit.col, start=hit.start, end=None) for hit in hits]


def extend_ln_tails_only(hits: Sequence[HitObject], extension_s: float = 1.5) -> list[HitObject]:
    extended: list[HitObject] = []
    for hit in hits:
        if hit.end is not None and hit.end > hit.start + CFG.min_ln_len:
            extended.append(HitObject(col=hit.col, start=hit.start, end=hit.end + extension_s))
        else:
            extended.append(HitObject(col=hit.col, start=hit.start, end=hit.end))
    return extended


PERTURBATION_STABLE_FEATURES = [
    "density_level",
    "density_burst",
    "hold_occupancy",
    "ln_change_rate",
    "chord_ratio",
    "jack_excess",
    "jack_streak_exposure",
    "repeat_exact",
    "repeat_shift",
    "repeat_motion",
    "repeat_rhythm",
]


@dataclass(frozen=True)
class PerturbationExpectation:
    perturbation: str
    feature: str
    direction: str
    min_delta: float | None = None
    max_abs_delta: float | None = None
    reason: str = ""


PERTURBATION_EXPECTATIONS = [
    PerturbationExpectation("single_stream_mirror", "hand_balance_signed", "sign_flip", max_abs_delta=0.02, reason="mirror should flip signed hand balance"),
    PerturbationExpectation("single_stream_mirror", "density_level", "invariant", max_abs_delta=0.02, reason="mirror should keep density"),
    PerturbationExpectation("single_stream_mirror", "chord_ratio", "invariant", max_abs_delta=0.02, reason="mirror should keep chord ratio"),
    PerturbationExpectation("single_stream_mirror", "repeat_shift", "invariant", max_abs_delta=0.02, reason="mirror should keep shift-normalized recurrence"),
    PerturbationExpectation("single_stream_time_stretch_1p1", "density_level", "decrease", min_delta=0.03, reason="time stretch should lower density"),
    PerturbationExpectation("single_stream_time_stretch_1p1", "chord_ratio", "invariant", max_abs_delta=0.02, reason="time stretch should not change chord mix"),
    PerturbationExpectation("normal_jack_time_stretch_1p1", "jack_excess", "not_increase", min_delta=0.02, reason="time stretch should not raise jack excess"),
    PerturbationExpectation("random_stream_column_shuffle", "density_level", "invariant", max_abs_delta=0.02, reason="column shuffle should keep onset density"),
    PerturbationExpectation("random_stream_column_shuffle", "chord_ratio", "invariant", max_abs_delta=0.02, reason="column shuffle preserves chord size"),
    PerturbationExpectation("normal_jack_column_shuffle", "density_level", "invariant", max_abs_delta=0.02, reason="column shuffle should keep onset density"),
    PerturbationExpectation("normal_jack_column_shuffle", "chord_ratio", "invariant", max_abs_delta=0.02, reason="column shuffle preserves chord size"),
    PerturbationExpectation("normal_jack_column_shuffle", "jack_excess", "decrease", min_delta=0.10, reason="column shuffle should reduce same-column jack excess"),
    PerturbationExpectation("normal_jack_column_shuffle", "repeat_exact", "decrease", min_delta=0.10, reason="column shuffle should reduce literal recurrence"),
    PerturbationExpectation("double_stair_event_order_randomize", "density_level", "invariant", max_abs_delta=0.02, reason="event order randomization keeps timestamps"),
    PerturbationExpectation("double_stair_event_order_randomize", "chord_ratio", "invariant", max_abs_delta=0.02, reason="event order randomization keeps chord sizes"),
    PerturbationExpectation("double_stair_event_order_randomize", "repeat_shift", "decrease", min_delta=0.05, reason="event order randomization should reduce shift recurrence"),
    PerturbationExpectation("double_stair_event_order_randomize", "repeat_motion", "decrease", min_delta=0.005, reason="event order randomization should reduce motion recurrence"),
    PerturbationExpectation("single_stream_singles_to_doubles", "density_level", "increase", min_delta=0.05, reason="singles-to-doubles should raise weighted density"),
    PerturbationExpectation("single_stream_singles_to_doubles", "chord_ratio", "increase", min_delta=0.10, reason="singles-to-doubles should raise chord ratio"),
    PerturbationExpectation("single_stream_singles_to_doubles", "jack_excess", "not_increase", min_delta=0.02, reason="singles-to-doubles should not create jack excess by itself"),
    PerturbationExpectation("normal_roll_force_same_column", "jack_excess", "increase", min_delta=0.10, reason="forcing a roll into one column should raise jack excess"),
    PerturbationExpectation("normal_roll_force_same_column", "repeat_exact", "increase", min_delta=0.10, reason="forcing a roll into one column should raise exact recurrence"),
    PerturbationExpectation("mixed_remove_ln_tails", "hold_occupancy", "decrease", min_delta=0.10, reason="removing LN tails should reduce hold occupancy"),
    PerturbationExpectation("mixed_remove_ln_tails", "density_level", "invariant", max_abs_delta=0.05, reason="removing tails should keep onset density"),
    PerturbationExpectation("mixed_remove_ln_tails", "chord_ratio", "invariant", max_abs_delta=0.05, reason="removing tails should keep onset chord ratio"),
    PerturbationExpectation("mixed_extend_ln_tails", "hold_occupancy", "increase", min_delta=0.04, reason="extending LN tails should increase hold occupancy"),
]


def perturbation_delta_table(
    base_hits: Sequence[HitObject],
    perturbed_hits: Sequence[HitObject],
    *,
    duration_s: float,
    features: Sequence[str] = MODEL_CHANNELS,
    summary_stat: str = "p95",
    beat_length_at: Callable[[float], float] = default_beat_length_at,
    perturbed_beat_length_at: Callable[[float], float] | None = None,
) -> pd.DataFrame:
    if perturbed_beat_length_at is None:
        perturbed_beat_length_at = beat_length_at
    base_out, base_frame = extract_v2_from_hits(
        base_hits,
        beat_length_at=beat_length_at,
        start_time=0.0,
        end_time=duration_s,
        include_debug=True,
    )
    pert_out, pert_frame = extract_v2_from_hits(
        perturbed_hits,
        beat_length_at=perturbed_beat_length_at,
        start_time=0.0,
        end_time=duration_s,
        include_debug=True,
    )
    summary_features = list(dict.fromkeys([*features, *CONFIDENCE_FEATURES]))
    base_summary = feature_summary_row(base_frame, summary_features)
    pert_summary = feature_summary_row(pert_frame, summary_features)
    rows = []
    for feature in features:
        column = f"{feature}_{summary_stat}"
        if column not in base_summary or column not in pert_summary:
            continue
        base_value = float(base_summary[column])
        pert_value = float(pert_summary[column])
        rows.append({
            "feature": feature,
            "summary_stat": summary_stat,
            "base": base_value,
            "perturbed": pert_value,
            "delta": pert_value - base_value,
            "abs_delta": abs(pert_value - base_value),
        })
    return pd.DataFrame(rows).sort_values("abs_delta", ascending=False).reset_index(drop=True)


def time_stretch_delta_table(
    hits: Sequence[HitObject],
    *,
    factor: float = 1.1,
    duration_s: float,
    beat_length_at: Callable[[float], float] = default_beat_length_at,
    features: Sequence[str] = MODEL_CHANNELS,
    summary_stat: str = "p95",
) -> pd.DataFrame:
    return perturbation_delta_table(
        hits,
        time_stretch(hits, factor),
        duration_s=duration_s * factor,
        features=features,
        summary_stat=summary_stat,
        beat_length_at=beat_length_at,
        perturbed_beat_length_at=stretch_beat_length_fn(beat_length_at, factor),
    )


def mirror_invariance_report(
    hits: Sequence[HitObject],
    *,
    duration_s: float = 16.0,
    features: Sequence[str] = PERTURBATION_STABLE_FEATURES,
) -> pd.DataFrame:
    grid = np.arange(0.0, duration_s + 0.5 * CFG.grid_step, CFG.grid_step)
    base_out, base_frame = extract_v2_from_hits(hits, grid=grid, include_debug=True)
    mirror_out, mirror_frame = extract_v2_from_hits(mirror_columns(hits), grid=grid, include_debug=True)
    rows = []
    for feature in features:
        base = base_frame[feature].to_numpy(dtype=float)
        mirrored = mirror_frame[feature].to_numpy(dtype=float)
        rows.append({
            "feature": feature,
            "invariance_error": float(np.mean(np.abs(base - mirrored))) if len(base) else 0.0,
            "abs_delta": float(np.mean(np.abs(base - mirrored))) if len(base) else 0.0,
        })
    hand_error = float(np.mean(np.abs(
        base_frame["hand_balance_signed"].to_numpy(dtype=float)
        + mirror_frame["hand_balance_signed"].to_numpy(dtype=float)
    )))
    rows.append({"feature": "hand_balance_signed", "invariance_error": hand_error, "abs_delta": hand_error, "expected": "sign_flip"})
    return pd.DataFrame(rows).sort_values("invariance_error", ascending=False).reset_index(drop=True)


def evaluate_perturbation_expectations(
    tables: dict[str, pd.DataFrame],
    expectations: Sequence[PerturbationExpectation] = PERTURBATION_EXPECTATIONS,
) -> pd.DataFrame:
    rows = []
    for item in expectations:
        table = tables.get(item.perturbation)
        if table is None or "feature" not in table:
            rows.append({**item.__dict__, "delta": math.nan, "abs_delta": math.nan, "pass": False, "failure": "missing perturbation table"})
            continue
        matched = table.loc[table["feature"].eq(item.feature)]
        if matched.empty:
            rows.append({**item.__dict__, "delta": math.nan, "abs_delta": math.nan, "pass": False, "failure": "missing feature row"})
            continue
        row = matched.iloc[0]
        delta = float(row["delta"]) if "delta" in row and pd.notna(row["delta"]) else math.nan
        abs_delta = float(row["abs_delta"]) if "abs_delta" in row and pd.notna(row["abs_delta"]) else float(row.get("invariance_error", math.nan))
        passed = True
        failure = ""
        if item.direction == "increase":
            threshold = 0.0 if item.min_delta is None else item.min_delta
            passed = bool(np.isfinite(delta) and delta >= threshold)
            failure = f"delta {delta:.4f} < {threshold:.4f}" if not passed else ""
        elif item.direction == "decrease":
            threshold = 0.0 if item.min_delta is None else item.min_delta
            passed = bool(np.isfinite(delta) and delta <= -threshold)
            failure = f"delta {delta:.4f} > -{threshold:.4f}" if not passed else ""
        elif item.direction == "not_increase":
            tolerance = 0.0 if item.min_delta is None else item.min_delta
            passed = bool(np.isfinite(delta) and delta <= tolerance)
            failure = f"delta {delta:.4f} > {tolerance:.4f}" if not passed else ""
        elif item.direction == "not_decrease":
            tolerance = 0.0 if item.min_delta is None else item.min_delta
            passed = bool(np.isfinite(delta) and delta >= -tolerance)
            failure = f"delta {delta:.4f} < -{tolerance:.4f}" if not passed else ""
        elif item.direction in {"invariant", "sign_flip"}:
            threshold = math.inf if item.max_abs_delta is None else item.max_abs_delta
            passed = bool(np.isfinite(abs_delta) and abs_delta <= threshold)
            failure = f"abs_delta {abs_delta:.4f} > {threshold:.4f}" if not passed else ""
        else:
            raise ValueError(f"unknown perturbation expectation direction: {item.direction}")
        if passed and item.max_abs_delta is not None and item.direction not in {"invariant", "sign_flip"}:
            passed = bool(abs_delta <= item.max_abs_delta)
            failure = f"abs_delta {abs_delta:.4f} > {item.max_abs_delta:.4f}" if not passed else ""
        rows.append({
            "perturbation": item.perturbation,
            "feature": item.feature,
            "direction": item.direction,
            "delta": delta,
            "abs_delta": abs_delta,
            "min_delta": item.min_delta,
            "max_abs_delta": item.max_abs_delta,
            "pass": passed,
            "failure": failure,
            "reason": item.reason,
        })
    return pd.DataFrame(rows)


def perturbation_suite(
    fixture_name: str,
    *,
    duration_s: float = 16.0,
    seed: int = 7,
) -> dict[str, pd.DataFrame]:
    if fixture_name not in SYNTHETIC_FIXTURES:
        raise KeyError(fixture_name)
    hits = SYNTHETIC_FIXTURES[fixture_name](duration_s)
    return {
        "mirror": mirror_invariance_report(hits, duration_s=duration_s),
        "time_stretch_1p1": time_stretch_delta_table(hits, duration_s=duration_s, factor=1.1),
        "column_shuffle": perturbation_delta_table(hits, column_shuffle_preserving_chord_size(hits, seed=seed), duration_s=duration_s),
        "event_order_randomize": perturbation_delta_table(hits, randomize_event_order_within_section(hits, seed=seed), duration_s=duration_s),
        "force_same_column": perturbation_delta_table(hits, force_same_column_repeats(hits), duration_s=duration_s),
        "remove_ln_tails": perturbation_delta_table(hits, remove_ln_tails(hits), duration_s=duration_s),
        "extend_ln_tails": perturbation_delta_table(hits, extend_ln_tails_only(hits), duration_s=duration_s + 1.5),
        "singles_to_doubles": perturbation_delta_table(hits, convert_singles_to_doubles(hits), duration_s=duration_s),
    }


## Phase 4: Dataset Audit

The dataset pass works at section level, not raw timepoint level. A section row summarizes one overlapping
4s/8s window and keeps both model channels and diagnostics such as `n_eff`, null-baseline terms, and
confidence. This is the table used for distribution checks, partial correlations, outlier mining, and
clustering.


In [8]:
SECTION_SUMMARY_STATS = ("min", "mean", "std", "p20", "p50", "p80", "p90", "p95", "max", "duration_above_threshold", "slope", "local_variance")

FEATURE_THRESHOLDS = {
    "density_burst": 0.50,
    "hold_occupancy": 0.50,
    "ln_change_rate": 0.50,
    "chord_ratio": 0.50,
    "jack_excess": 0.50,
    "jack_streak_exposure": 0.50,
    "hand_imbalance_abs": 0.50,
    "repeat_exact": 0.50,
    "repeat_shift": 0.50,
    "repeat_motion": 0.50,
    "repeat_rhythm": 0.50,
}


def _numeric_summary_columns(frame: pd.DataFrame) -> list[str]:
    columns: list[str] = []
    for column in frame.columns:
        if column == "time_s":
            continue
        if pd.api.types.is_numeric_dtype(frame[column]) or pd.api.types.is_bool_dtype(frame[column]):
            columns.append(column)
    return columns


def _section_slope(times: np.ndarray, values: np.ndarray) -> float:
    if len(values) < 2:
        return 0.0
    centered = times - float(np.mean(times))
    denom = float(np.sum(centered * centered))
    if denom <= 1e-12:
        return 0.0
    return float(np.sum(centered * (values - float(np.mean(values)))) / denom)


def summarize_numeric_column(times: np.ndarray, values: np.ndarray, *, threshold: float, grid_step: float) -> dict[str, float]:
    finite = np.isfinite(values)
    values = values[finite]
    times = times[finite]
    if len(values) == 0:
        return {stat: 0.0 for stat in SECTION_SUMMARY_STATS}
    return {
        "min": float(np.min(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "p20": float(np.percentile(values, 20)),
        "p50": float(np.percentile(values, 50)),
        "p80": float(np.percentile(values, 80)),
        "p90": float(np.percentile(values, 90)),
        "p95": float(np.percentile(values, 95)),
        "max": float(np.max(values)),
        "duration_above_threshold": float(np.sum(values >= threshold) * grid_step),
        "slope": _section_slope(times, values),
        "local_variance": float(np.var(values)),
    }


def append_peak_diagnostics(record: dict[str, Any], section_for_values: pd.DataFrame) -> None:
    for feature, confidence in FEATURE_CONFIDENCE_MAP.items():
        if feature not in section_for_values.columns:
            continue
        values = pd.to_numeric(section_for_values[feature], errors="coerce").to_numpy(dtype=float)
        finite = np.isfinite(values)
        if not finite.any():
            continue
        finite_positions = np.flatnonzero(finite)
        peak_pos = int(finite_positions[np.argmax(values[finite])])
        record[f"{feature}_peak_time_s"] = float(section_for_values["time_s"].iloc[peak_pos])
        record[f"{feature}_peak_value"] = float(values[peak_pos])
        if confidence in section_for_values.columns:
            confidence_values = pd.to_numeric(section_for_values[confidence], errors="coerce").to_numpy(dtype=float)
            record[f"{feature}_confidence_at_peak"] = float(confidence_values[peak_pos]) if np.isfinite(confidence_values[peak_pos]) else math.nan
        for label, column in PEAK_DEBUG_COLUMNS.get(feature, {}).items():
            if column not in section_for_values.columns:
                continue
            debug_values = pd.to_numeric(section_for_values[column], errors="coerce").to_numpy(dtype=float)
            record[f"{feature}_{label}_at_peak"] = float(debug_values[peak_pos]) if np.isfinite(debug_values[peak_pos]) else math.nan


def section_summaries_for_frame(
    row: pd.Series | None,
    frame: pd.DataFrame,
    *,
    section_s: float = 8.0,
    stride_s: float = 4.0,
    bpm_median: float = math.nan,
    map_duration_s: float | None = None,
) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()
    duration = float(map_duration_s if map_duration_s is not None else frame["time_s"].max())
    starts = np.arange(0.0, max(0.0, duration - section_s) + 0.5 * stride_s, stride_s)
    numeric_columns = _numeric_summary_columns(frame)
    records: list[dict[str, Any]] = []
    grid_step = float(np.median(np.diff(frame["time_s"].to_numpy(dtype=float)))) if len(frame) > 1 else CFG.grid_step

    for start_s in starts:
        end_s = start_s + section_s
        section = frame.loc[(frame["time_s"] >= start_s) & (frame["time_s"] < end_s)].copy()
        if section.empty:
            continue
        if "valid_control_mask" in section.columns:
            valid = section["valid_control_mask"].to_numpy(dtype=bool)
            valid_fraction = float(np.mean(valid)) if len(valid) else 0.0
            section_for_values = section.loc[valid].copy()
            if section_for_values.empty:
                continue
        else:
            valid_fraction = 1.0
            section_for_values = section

        record: dict[str, Any] = {
            "section_start_s": float(start_s),
            "section_end_s": float(end_s),
            "section_center_s": float(start_s + 0.5 * section_s),
            "section_s": float(section_s),
            "stride_s": float(stride_s),
            "rows": int(len(section)),
            "valid_rows": int(len(section_for_values)),
            "valid_fraction": valid_fraction,
            "bpm_median": float(bpm_median),
            "map_duration_s": float(duration),
        }
        if row is not None:
            for column in ["filtered_index", "beatmap_id", "difficulty", "artist", "title", "version", "creator"]:
                if column in row:
                    value = row[column]
                    if isinstance(value, np.generic):
                        value = value.item()
                    record[column] = value

        times = section_for_values["time_s"].to_numpy(dtype=float)
        for column in numeric_columns:
            if column == "valid_control_mask":
                record["valid_control_mask_mean"] = float(np.mean(section[column].to_numpy(dtype=bool)))
                continue
            values = section_for_values[column].to_numpy(dtype=float)
            threshold = FEATURE_THRESHOLDS.get(column, 0.50)
            stats = summarize_numeric_column(times, values, threshold=threshold, grid_step=grid_step)
            for stat, value in stats.items():
                record[f"{column}_{stat}"] = value
        append_peak_diagnostics(record, section_for_values)
        records.append(record)
    return pd.DataFrame(records)


def _sample_metadata(
    *,
    sample_size: int | None,
    seed: int,
    difficulty_min: float | None = None,
    difficulty_max: float | None = None,
) -> pd.DataFrame:
    df = metadata.copy()
    if difficulty_min is not None:
        df = df.loc[df["difficulty"] >= difficulty_min]
    if difficulty_max is not None:
        df = df.loc[df["difficulty"] <= difficulty_max]
    if sample_size is not None and sample_size < len(df):
        df = df.sample(n=sample_size, random_state=seed)
    return df.sort_values("filtered_index").reset_index(drop=True)


def build_dataset_section_audit(
    *,
    sample_size: int | None = 256,
    seed: int = 7,
    section_s: float = 8.0,
    stride_s: float = 4.0,
    output_path: Path | None = SECTION_AUDIT_PATH,
    difficulty_min: float | None = None,
    difficulty_max: float | None = None,
    progress_every: int = 25,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    selected = _sample_metadata(
        sample_size=sample_size,
        seed=seed,
        difficulty_min=difficulty_min,
        difficulty_max=difficulty_max,
    )
    records: list[pd.DataFrame] = []
    errors: list[dict[str, Any]] = []
    for i, row in selected.iterrows():
        try:
            inputs, out, frame = extract_v2_for_row(row, include_debug=True)
            sections = section_summaries_for_frame(
                row,
                frame,
                section_s=section_s,
                stride_s=stride_s,
                bpm_median=inputs.bpm_median,
                map_duration_s=inputs.map_duration_s,
            )
            if not sections.empty:
                records.append(sections)
        except Exception as exc:
            errors.append({
                "filtered_index": int(row["filtered_index"]),
                "beatmap_id": int(row["beatmap_id"]),
                "error_type": type(exc).__name__,
                "error": str(exc),
            })
        if progress_every and (i + 1) % progress_every == 0:
            print(f"processed {i + 1}/{len(selected)} maps")

    section_df = pd.concat(records, ignore_index=True) if records else pd.DataFrame()
    error_df = pd.DataFrame(errors)
    if output_path is not None:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        section_df.to_parquet(output_path, index=False)
        error_path = output_path.with_name(output_path.stem + "_errors.parquet")
        error_df.to_parquet(error_path, index=False)
        print("wrote", output_path)
        print("wrote", error_path)
    return section_df, error_df


TIMESERIES_METADATA_COLUMNS = [
    "filtered_index",
    "source_index",
    "beatmap_id",
    "beatmap_set_id",
    "difficulty",
    "time_s",
]


def _timeseries_feature_frame(group: pd.DataFrame) -> pd.DataFrame:
    columns = [
        column
        for column in group.columns
        if column == "time_s" or column not in TIMESERIES_METADATA_COLUMNS
    ]
    return group.loc[:, columns].sort_values("time_s").reset_index(drop=True)


def _timeseries_metadata_row(group: pd.DataFrame) -> pd.Series:
    filtered_index = int(group["filtered_index"].iloc[0])
    metadata_frame = globals().get("metadata")
    if isinstance(metadata_frame, pd.DataFrame) and "filtered_index" in metadata_frame.columns:
        matched = metadata_frame.loc[metadata_frame["filtered_index"].eq(filtered_index)]
        if not matched.empty:
            return matched.iloc[0]
    fallback = {
        column: group[column].iloc[0]
        for column in TIMESERIES_METADATA_COLUMNS
        if column in group.columns and column != "time_s"
    }
    return pd.Series(fallback)


def build_section_audit_from_timeseries_parquet(
    *,
    timeseries_path: Path = TIMESERIES_PATH,
    sample_size: int | None = None,
    seed: int = 7,
    section_s: float = 8.0,
    stride_s: float = 4.0,
    output_path: Path | None = SECTION_AUDIT_PATH,
    filtered_indexes: Sequence[int] | None = None,
    progress_every: int = 25,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    timeseries_path = Path(timeseries_path)
    if not timeseries_path.exists():
        raise FileNotFoundError(timeseries_path)

    time_df = pd.read_parquet(timeseries_path)
    required = {"filtered_index", "time_s", "valid_control_mask"}
    missing = sorted(required - set(time_df.columns))
    if missing:
        raise RuntimeError(f"timeseries parquet missing required columns: {missing}")

    if filtered_indexes is not None:
        selected_indexes = sorted({int(index) for index in filtered_indexes})
        time_df = time_df.loc[time_df["filtered_index"].isin(selected_indexes)].copy()
    elif sample_size is not None:
        unique_indexes = np.asarray(sorted(time_df["filtered_index"].dropna().astype(int).unique()), dtype=int)
        if sample_size < len(unique_indexes):
            rng = np.random.default_rng(seed)
            selected_indexes = sorted(rng.choice(unique_indexes, size=sample_size, replace=False).astype(int).tolist())
            time_df = time_df.loc[time_df["filtered_index"].isin(selected_indexes)].copy()

    records: list[pd.DataFrame] = []
    errors: list[dict[str, Any]] = []
    grouped = time_df.sort_values(["filtered_index", "time_s"]).groupby("filtered_index", sort=True)
    total_maps = int(grouped.ngroups)
    for position, (filtered_index, group) in enumerate(grouped, start=1):
        try:
            row = _timeseries_metadata_row(group)
            frame = _timeseries_feature_frame(group)
            map_duration_s = float(frame["time_s"].max()) if not frame.empty else 0.0
            sections = section_summaries_for_frame(
                row,
                frame,
                section_s=section_s,
                stride_s=stride_s,
                bpm_median=math.nan,
                map_duration_s=map_duration_s,
            )
            if not sections.empty:
                records.append(sections)
        except Exception as exc:
            beatmap_id = int(group["beatmap_id"].iloc[0]) if "beatmap_id" in group else -1
            errors.append({
                "filtered_index": int(filtered_index),
                "beatmap_id": beatmap_id,
                "error_type": type(exc).__name__,
                "error": str(exc),
            })
        if progress_every and position % progress_every == 0:
            print(f"processed parquet maps {position}/{total_maps}")

    section_df = pd.concat(records, ignore_index=True) if records else pd.DataFrame()
    error_df = pd.DataFrame(errors)
    if output_path is not None:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        section_df.to_parquet(output_path, index=False)
        error_path = output_path.with_name(output_path.stem + "_errors.parquet")
        error_df.to_parquet(error_path, index=False)
        print("wrote", output_path)
        print("wrote", error_path)
    return section_df, error_df


def load_or_build_section_audit_from_timeseries(
    *,
    path: Path = SECTION_AUDIT_PATH,
    timeseries_path: Path = TIMESERIES_PATH,
    rebuild: bool = False,
    sample_size: int | None = None,
    seed: int = 7,
) -> pd.DataFrame:
    if path.exists() and not rebuild:
        return pd.read_parquet(path)
    section_df, error_df = build_section_audit_from_timeseries_parquet(
        timeseries_path=timeseries_path,
        sample_size=sample_size,
        seed=seed,
        output_path=path,
    )
    if not error_df.empty:
        display(error_df.head(20))
    return section_df


def load_or_build_section_audit(
    *,
    path: Path = SECTION_AUDIT_PATH,
    rebuild: bool = False,
    sample_size: int | None = 256,
    seed: int = 7,
) -> pd.DataFrame:
    if path.exists() and not rebuild:
        return pd.read_parquet(path)
    section_df, error_df = build_dataset_section_audit(sample_size=sample_size, seed=seed, output_path=path)
    if not error_df.empty:
        display(error_df.head(20))
    return section_df


In [9]:
def distribution_report(
    section_df: pd.DataFrame,
    *,
    features: Sequence[str] = MODEL_CHANNELS,
    stat: str = "mean",
) -> pd.DataFrame:
    rows = []
    for feature in features:
        column = f"{feature}_{stat}"
        if column not in section_df:
            continue
        values = pd.to_numeric(section_df[column], errors="coerce").dropna().to_numpy(dtype=float)
        rows.append({
            "feature": feature,
            "count": int(len(values)),
            "mean": float(np.mean(values)) if len(values) else math.nan,
            "std": float(np.std(values)) if len(values) else math.nan,
            "p50": float(np.percentile(values, 50)) if len(values) else math.nan,
            "p80": float(np.percentile(values, 80)) if len(values) else math.nan,
            "p95": float(np.percentile(values, 95)) if len(values) else math.nan,
            "p99": float(np.percentile(values, 99)) if len(values) else math.nan,
            "max": float(np.max(values)) if len(values) else math.nan,
        })
    return pd.DataFrame(rows)


def saturation_report(
    section_df: pd.DataFrame,
    *,
    features: Sequence[str] = VALUE_FEATURES,
    stat: str = "p95",
    thresholds: dict[str, tuple[float, float]] = SATURATION_THRESHOLDS,
) -> pd.DataFrame:
    rows = []
    for feature in features:
        column = f"{feature}_{stat}"
        if column not in section_df:
            continue
        values = pd.to_numeric(section_df[column], errors="coerce").dropna().to_numpy(dtype=float)
        p95 = float(np.percentile(values, 95)) if len(values) else math.nan
        p99 = float(np.percentile(values, 99)) if len(values) else math.nan
        max_value = float(np.max(values)) if len(values) else math.nan
        row = {
            "feature": feature,
            "stat": stat,
            "count": int(len(values)),
            "p95": p95,
            "p99": p99,
            "max": max_value,
            "near_zero_rate": math.nan,
            "near_high_rate": math.nan,
            "tail_heaviness": math.nan,
            "max_to_p99": math.nan,
        }
        if feature in thresholds:
            low_threshold, high_threshold = thresholds[feature]
            row.update({
                "low_threshold": low_threshold,
                "high_threshold": high_threshold,
                "near_zero_rate": float(np.mean(values <= low_threshold)) if len(values) else math.nan,
                "near_high_rate": float(np.mean(values >= high_threshold)) if len(values) else math.nan,
            })
        else:
            row.update({
                "low_threshold": math.nan,
                "high_threshold": math.nan,
                "tail_heaviness": p99 / max(abs(p95), 1e-9) if np.isfinite(p99) and np.isfinite(p95) else math.nan,
                "max_to_p99": max_value / max(abs(p99), 1e-9) if np.isfinite(max_value) and np.isfinite(p99) else math.nan,
            })
        rows.append(row)
    if not rows:
        return pd.DataFrame()
    out = pd.DataFrame(rows)
    out["sort_score"] = out["near_high_rate"].fillna(0.0) + out["tail_heaviness"].fillna(0.0)
    return out.sort_values("sort_score", ascending=False).drop(columns=["sort_score"]).reset_index(drop=True)


def _numeric_column_or_nan(df: pd.DataFrame, column: str) -> pd.Series:
    if column in df:
        return pd.to_numeric(df[column], errors="coerce")
    return pd.Series(math.nan, index=df.index, dtype=float)


def low_confidence_high_value_report(
    section_df: pd.DataFrame,
    *,
    value_threshold: float = 0.50,
    confidence_threshold: float = 0.20,
    n: int = 100,
) -> pd.DataFrame:
    rows = []
    for feature, confidence in FEATURE_CONFIDENCE_MAP.items():
        value_col = f"{feature}_p95"
        if value_col not in section_df:
            continue
        value = pd.to_numeric(section_df[value_col], errors="coerce")
        confidence_mean = _numeric_column_or_nan(section_df, f"{confidence}_mean")
        confidence_p20 = _numeric_column_or_nan(section_df, f"{confidence}_p20")
        confidence_min = _numeric_column_or_nan(section_df, f"{confidence}_min")
        confidence_at_peak = _numeric_column_or_nan(section_df, f"{feature}_confidence_at_peak")
        low_confidence = pd.concat(
            [confidence_mean, confidence_p20, confidence_min, confidence_at_peak],
            axis=1,
        ).le(confidence_threshold).any(axis=1)
        mask = (value >= value_threshold) & low_confidence
        selected = section_df.loc[mask].copy()
        if selected.empty:
            continue
        selected["feature"] = feature
        selected["value"] = value.loc[selected.index]
        selected["confidence"] = confidence_mean.loc[selected.index]
        selected["confidence_mean"] = confidence_mean.loc[selected.index]
        selected["confidence_p20"] = confidence_p20.loc[selected.index]
        selected["confidence_min"] = confidence_min.loc[selected.index]
        selected["confidence_at_feature_peak"] = confidence_at_peak.loc[selected.index]
        rows.append(selected)
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True).sort_values(["feature", "value"], ascending=[True, False]).head(n)
    keep = [
        "feature",
        "value",
        "confidence",
        "confidence_mean",
        "confidence_p20",
        "confidence_min",
        "confidence_at_feature_peak",
        "valid_fraction",
        "control_confidence_mean",
        "filtered_index",
        "beatmap_id",
        "difficulty",
        "artist",
        "title",
        "version",
        "section_start_s",
        "section_end_s",
    ]
    keep = [column for column in keep if column in out.columns]
    return out[keep].reset_index(drop=True)


def feature_correlation_matrix(
    section_df: pd.DataFrame,
    *,
    features: Sequence[str] = MODEL_CHANNELS,
    stat: str = "mean",
    method: str = "pearson",
) -> pd.DataFrame:
    if method not in {"pearson", "spearman"}:
        raise ValueError("method must be 'pearson' or 'spearman'")
    columns = [f"{feature}_{stat}" for feature in features if f"{feature}_{stat}" in section_df]
    return section_df[columns].corr(method=method)


def feature_correlation_reports(
    section_df: pd.DataFrame,
    *,
    features: Sequence[str] = MODEL_CHANNELS,
    stat: str = "mean",
) -> dict[str, pd.DataFrame]:
    return {
        "pearson": feature_correlation_matrix(section_df, features=features, stat=stat, method="pearson"),
        "spearman": feature_correlation_matrix(section_df, features=features, stat=stat, method="spearman"),
    }


def _residualize(y: np.ndarray, controls: np.ndarray) -> np.ndarray:
    finite = np.isfinite(y) & np.all(np.isfinite(controls), axis=1)
    out = np.full_like(y, np.nan, dtype=float)
    if finite.sum() < controls.shape[1] + 2:
        return out
    x = np.column_stack([np.ones(finite.sum()), controls[finite]])
    beta, *_ = np.linalg.lstsq(x, y[finite], rcond=None)
    out[finite] = y[finite] - x @ beta
    return out


def partial_correlation(
    section_df: pd.DataFrame,
    x_col: str,
    y_col: str,
    control_cols: Sequence[str] = ("density_level_mean",),
) -> float:
    columns = [x_col, y_col, *control_cols]
    data = section_df[columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    x = data[:, 0]
    y = data[:, 1]
    controls = data[:, 2:]
    rx = _residualize(x, controls)
    ry = _residualize(y, controls)
    finite = np.isfinite(rx) & np.isfinite(ry)
    if finite.sum() < 3:
        return math.nan
    return float(np.corrcoef(rx[finite], ry[finite])[0, 1])


def selected_partial_correlations(section_df: pd.DataFrame) -> pd.DataFrame:
    pairs = [
        ("jack_excess_mean", "density_level_mean", ("chord_ratio_mean",)),
        ("jack_excess_mean", "density_level_mean", ("density_raw_med_mean",)),
        ("repeat_motion_mean", "density_level_mean", ("chord_ratio_mean",)),
        ("repeat_exact_mean", "jack_excess_mean", ("density_level_mean",)),
        ("ln_change_rate_mean", "hold_occupancy_mean", ("density_level_mean",)),
        ("chord_ratio_mean", "density_level_mean", ()),
        ("repeat_shift_mean", "repeat_motion_mean", ("density_level_mean",)),
        ("hand_imbalance_abs_mean", "density_level_mean", ()),
    ]
    rows = []
    for x, y, preferred_controls in pairs:
        if x not in section_df or y not in section_df:
            continue
        pearson = section_df[[x, y]].corr(method="pearson").iloc[0, 1]
        spearman = section_df[[x, y]].corr(method="spearman").iloc[0, 1]
        controls = tuple(col for col in preferred_controls if col not in {x, y} and col in section_df)
        partial = partial_correlation(section_df, x, y, controls) if controls else math.nan
        rows.append({"x": x, "y": y, "pearson": pearson, "spearman": spearman, "controls": ",".join(controls), "partial": partial})
    return pd.DataFrame(rows)


def _audit_gate_row(gate_name: str, metric: str, value: float, threshold: str, passed: bool, reason: str) -> dict[str, Any]:
    return {
        "gate_name": gate_name,
        "metric": metric,
        "value": float(value) if np.isfinite(value) else math.nan,
        "threshold": threshold,
        "pass": bool(passed),
        "reason": reason,
    }


def _finite_rate(section_df: pd.DataFrame, columns: Sequence[str]) -> float:
    available = [column for column in columns if column in section_df]
    if not available or section_df.empty:
        return math.nan
    values = section_df[available].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    return float(np.isfinite(values).mean())


def evaluate_control_v2_audit(section_df: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    if section_df.empty:
        return pd.DataFrame([
            _audit_gate_row("section_count", "rows", 0.0, "> 0", False, "dataset audit produced no section rows")
        ])

    model_stat_columns = [f"{feature}_mean" for feature in MODEL_CHANNELS]
    finite_rate = _finite_rate(section_df, model_stat_columns)
    rows.append(_audit_gate_row("finite_model_stats", "finite_rate", finite_rate, ">= 0.999", finite_rate >= 0.999, "model section statistics should be finite"))

    if "valid_fraction" in section_df:
        valid_mean = float(pd.to_numeric(section_df["valid_fraction"], errors="coerce").mean())
        rows.append(_audit_gate_row("valid_fraction", "mean", valid_mean, ">= 0.80", valid_mean >= 0.80, "section windows should mostly cover valid map span"))

    low_conf = low_confidence_high_value_report(section_df, n=max(1, len(section_df) * len(FEATURE_CONFIDENCE_MAP)))
    low_conf_rate = len(low_conf) / max(1, len(section_df))
    rows.append(_audit_gate_row("low_confidence_high_value", "sections_per_input_section", low_conf_rate, "<= 0.10", low_conf_rate <= 0.10, "high value channels should not usually occur at low confidence"))

    sat = saturation_report(section_df)
    for feature in ["chord_ratio", "repeat_exact", "repeat_shift", "repeat_motion", "repeat_rhythm"]:
        matched = sat.loc[sat["feature"].eq(feature)] if not sat.empty else pd.DataFrame()
        if matched.empty:
            continue
        near_high = float(matched.iloc[0].get("near_high_rate", math.nan))
        near_zero = float(matched.iloc[0].get("near_zero_rate", math.nan))
        rows.append(_audit_gate_row(f"{feature}_near_high", "near_high_rate", near_high, "<= 0.05", near_high <= 0.05, "bounded channels should not saturate high across the corpus"))
        if feature.startswith("repeat_"):
            rows.append(_audit_gate_row(f"{feature}_near_zero", "near_zero_rate", near_zero, "<= 0.98", near_zero <= 0.98, "repeat channels should not be structurally zero"))

    correlation_specs = [
        ("jack_density_given_chord", "jack_excess_mean", "density_level_mean", ("chord_ratio_mean",), 0.80),
        ("jack_density_given_raw_density", "jack_excess_mean", "density_level_mean", ("density_raw_med_mean",), 0.80),
        ("repeat_motion_density_given_chord", "repeat_motion_mean", "density_level_mean", ("chord_ratio_mean",), 0.85),
        ("repeat_exact_jack_given_density", "repeat_exact_mean", "jack_excess_mean", ("density_level_mean",), 0.85),
        ("ln_change_hold_given_density", "ln_change_rate_mean", "hold_occupancy_mean", ("density_level_mean",), 0.90),
    ]
    for gate_name, x_col, y_col, controls, threshold in correlation_specs:
        available_controls = tuple(col for col in controls if col in section_df)
        if x_col not in section_df or y_col not in section_df or len(available_controls) != len(controls):
            rows.append(_audit_gate_row(gate_name, "abs_partial_corr", math.nan, f"<= {threshold:.2f}", False, "required columns missing"))
            continue
        value = abs(partial_correlation(section_df, x_col, y_col, available_controls))
        rows.append(_audit_gate_row(gate_name, "abs_partial_corr", value, f"<= {threshold:.2f}", value <= threshold, "operator should not collapse into the controlled proxy"))

    return pd.DataFrame(rows)


def add_analysis_buckets(section_df: pd.DataFrame) -> pd.DataFrame:
    out = section_df.copy()
    if "difficulty" in out:
        out["star_bucket"] = pd.cut(out["difficulty"], bins=[2, 3, 4, 5, 6], include_lowest=True)
    if "bpm_median" in out:
        out["bpm_bucket"] = pd.cut(out["bpm_median"], bins=[0, 120, 160, 200, 260, math.inf])
    if "map_duration_s" in out:
        out["length_bucket"] = pd.cut(out["map_duration_s"], bins=[0, 90, 150, 240, math.inf])
    return out


def bucketed_feature_correlation(
    section_df: pd.DataFrame,
    feature_a: str,
    feature_b: str,
    *,
    bucket_col: str,
    stat: str = "mean",
    method: str = "spearman",
) -> pd.DataFrame:
    a_col = f"{feature_a}_{stat}"
    b_col = f"{feature_b}_{stat}"
    rows = []
    for bucket, group in section_df.groupby(bucket_col, observed=False):
        if len(group) < 3 or a_col not in group or b_col not in group:
            corr = math.nan
        else:
            corr = float(group[[a_col, b_col]].corr(method=method).iloc[0, 1])
        rows.append({"bucket": str(bucket), "rows": int(len(group)), "method": method, "corr": corr})
    return pd.DataFrame(rows)


## Phase 5: Outlier Mining

Outlier mining ranks sections, not maps. Use these helpers after building or loading the section audit.
`export_outlier_bundle(...)` writes local object/feature plots plus repeat-token dumps for manual review.


In [10]:
def top_sections_by_feature(
    section_df: pd.DataFrame,
    feature: str,
    *,
    metric: str = "p95",
    n: int = 100,
    ascending: bool = False,
) -> pd.DataFrame:
    column = f"{feature}_{metric}"
    if column not in section_df:
        raise KeyError(column)
    keep = [
        "filtered_index",
        "beatmap_id",
        "difficulty",
        "artist",
        "title",
        "version",
        "section_start_s",
        "section_end_s",
        "valid_fraction",
        "control_confidence_mean",
        column,
        f"{feature}_peak_time_s",
        f"{feature}_confidence_at_peak",
    ]
    keep = [col for col in keep if col in section_df]
    return section_df.sort_values(column, ascending=ascending)[keep].head(n).reset_index(drop=True)


def _section_numeric(df: pd.DataFrame, column: str, default: float = 0.0) -> pd.Series:
    if column in df:
        return pd.to_numeric(df[column], errors="coerce").fillna(default)
    return pd.Series(default, index=df.index, dtype=float)


def disagreement_sections(section_df: pd.DataFrame, kind: str, *, n: int = 100) -> pd.DataFrame:
    df = section_df.copy()
    density_rank = _section_numeric(df, "density_level_p95").rank(pct=True)
    jack_observed_rank = _section_numeric(df, "jack_observed_p95").rank(pct=True)
    if kind == "high_jack_excess_low_observed":
        df["score"] = _section_numeric(df, "jack_excess_p95") - _section_numeric(df, "jack_observed_p95")
    elif kind == "high_jack_observed_low_excess":
        df["score"] = _section_numeric(df, "jack_observed_p95") - _section_numeric(df, "jack_excess_p95")
    elif kind == "high_repeat_shift_low_motion":
        df["score"] = _section_numeric(df, "repeat_shift_p95") - _section_numeric(df, "repeat_motion_p95")
    elif kind == "high_exact_and_jack":
        df["score"] = _section_numeric(df, "repeat_exact_p95") + _section_numeric(df, "jack_excess_p95")
    elif kind == "high_chord_low_density":
        df["score"] = _section_numeric(df, "chord_ratio_p95") - density_rank
    elif kind == "high_hand_low_confidence":
        df["score"] = _section_numeric(df, "hand_imbalance_abs_p95") - _section_numeric(df, "hand_confidence_mean")
    elif kind == "high_repeat_low_confidence":
        repeat_peak = pd.concat([
            _section_numeric(df, "repeat_exact_p95"),
            _section_numeric(df, "repeat_shift_p95"),
            _section_numeric(df, "repeat_motion_p95"),
            _section_numeric(df, "repeat_rhythm_p95"),
        ], axis=1).max(axis=1)
        df["score"] = repeat_peak - _section_numeric(df, "repeat_confidence_mean")
    elif kind == "high_repeat_rhythm_low_density":
        df["score"] = _section_numeric(df, "repeat_rhythm_p95") - density_rank
    elif kind == "high_jack_high_density_low_excess":
        df["score"] = jack_observed_rank + density_rank - _section_numeric(df, "jack_excess_p95")
    elif kind == "high_ln_change_low_hold":
        df["score"] = _section_numeric(df, "ln_change_rate_p95") - _section_numeric(df, "hold_occupancy_p95")
    elif kind == "high_hold_low_ln_change":
        df["score"] = _section_numeric(df, "hold_occupancy_p95") - _section_numeric(df, "ln_change_rate_p95").rank(pct=True)
    elif kind == "high_density_burst_low_density_level":
        df["score"] = _section_numeric(df, "density_burst_p95") - density_rank
    elif kind == "high_control_confidence_low_valid_fraction":
        df["score"] = _section_numeric(df, "control_confidence_mean") - _section_numeric(df, "valid_fraction")
    else:
        raise ValueError(f"unknown disagreement kind: {kind}")

    keep = [
        "filtered_index",
        "beatmap_id",
        "difficulty",
        "artist",
        "title",
        "version",
        "section_start_s",
        "section_end_s",
        "valid_fraction",
        "control_confidence_mean",
        "score",
        "density_level_p95",
        "density_burst_p95",
        "hold_occupancy_p95",
        "ln_change_rate_p95",
        "chord_ratio_p95",
        "chord_ratio_confidence_at_peak",
        "jack_excess_p95",
        "jack_excess_confidence_at_peak",
        "jack_observed_p95",
        "repeat_exact_p95",
        "repeat_exact_confidence_at_peak",
        "repeat_shift_p95",
        "repeat_motion_p95",
        "repeat_rhythm_p95",
        "repeat_confidence_mean",
        "hand_imbalance_abs_p95",
        "hand_imbalance_abs_confidence_at_peak",
        "hand_confidence_mean",
    ]
    keep = [col for col in keep if col in df]
    return df.sort_values("score", ascending=False)[keep].head(n).reset_index(drop=True)


def local_diagnostics_for_section(section: pd.Series, *, window_s: float = 12.0) -> tuple[MapInputs, dict[str, Any], pd.DataFrame, float]:
    inputs, out, frame = extract_v2_for_row(int(section["filtered_index"]), include_debug=True)
    section_start = float(section["section_start_s"])
    section_end = float(section["section_end_s"])
    center = 0.5 * (section_start + section_end)
    plot_start = max(0.0, center - 0.5 * window_s)
    return inputs, out, frame, plot_start


def local_object_table_for_window(
    inputs: MapInputs,
    frame: pd.DataFrame,
    *,
    start_s: float,
    end_s: float,
) -> pd.DataFrame:
    time = frame["time_s"].to_numpy(dtype=float)
    valid = frame["valid_control_mask"].to_numpy(dtype=bool) if "valid_control_mask" in frame else np.ones(len(frame), dtype=bool)
    chord_sizes: dict[float, int] = Counter(round(hit.start, 6) for hit in inputs.hits)
    last_col_time: dict[int, float] = {}
    previous_time: float | None = None
    rows = []
    for hit in sorted(inputs.hits, key=lambda item: (item.start, item.col)):
        if not (start_s <= hit.start <= end_s):
            continue
        idx = int(np.searchsorted(time, hit.start, side="left")) if len(time) else 0
        candidates = [candidate for candidate in [idx - 1, idx] if 0 <= candidate < len(time)]
        nearest = min(candidates, key=lambda candidate: abs(time[candidate] - hit.start)) if candidates else None
        same_column_gap = hit.start - last_col_time[hit.col] if hit.col in last_col_time else math.nan
        gap = hit.start - previous_time if previous_time is not None else math.nan
        duration = (hit.end - hit.start) if hit.end is not None else 0.0
        rows.append({
            "time_s": float(hit.start),
            "col": int(hit.col),
            "duration_s": float(duration),
            "mask_at_onset": int(valid[nearest]) if nearest is not None else 0,
            "chord_size": int(chord_sizes.get(round(hit.start, 6), 1)),
            "gap_s": float(gap) if np.isfinite(gap) else math.nan,
            "same_column_gap_s": float(same_column_gap) if np.isfinite(same_column_gap) else math.nan,
        })
        last_col_time[hit.col] = hit.start
        previous_time = hit.start
    return pd.DataFrame(rows)


def export_outlier_bundle(
    sections: pd.DataFrame,
    *,
    label: str,
    n: int = 20,
    window_s: float = 12.0,
    output_dir: Path = OUTLIER_DIR,
) -> Path:
    output_dir = output_dir / label
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest_rows = []
    for rank, section in sections.head(n).reset_index(drop=True).iterrows():
        inputs, out, frame, plot_start = local_diagnostics_for_section(section, window_s=window_s)
        filtered_index = int(section["filtered_index"])
        stem = f"rank{rank:03d}_filtered{filtered_index}_t{plot_start:.1f}"
        image_path = output_dir / f"{stem}.png"
        save_local_section_plot(
            image_path,
            inputs,
            out,
            frame,
            start_s=plot_start,
            window_s=window_s,
            features=DIAGNOSTIC_PLOT_FEATURES,
        )
        token_dump = {
            f"{family}_top_token_over_timegrid": repeat_top_tokens_over_timegrid(
                out,
                start_s=plot_start,
                end_s=plot_start + window_s,
                family=family,
                top_n=10,
            ).to_dict(orient="records")
            for family in ["exact", "shift", "motion", "rhythm"]
        }
        token_path = output_dir / f"{stem}_top_token_over_timegrid.json"
        token_path.write_text(json.dumps(token_dump, indent=2), encoding="utf-8")
        object_path = output_dir / f"{stem}_objects.csv"
        local_object_table_for_window(
            inputs,
            frame,
            start_s=plot_start,
            end_s=plot_start + window_s,
        ).to_csv(object_path, index=False)
        manifest = section.to_dict()
        manifest["plot_path"] = str(image_path)
        manifest["token_path"] = str(token_path)
        manifest["object_path"] = str(object_path)
        manifest["token_source"] = "top_token_over_timegrid"
        manifest_rows.append(manifest)
    manifest_path = output_dir / "manifest.csv"
    pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False)
    return manifest_path


## Phase 6: Section Clustering

The clustering input is a section embedding with robust scaling. KMeans is implemented locally so the
notebook does not depend on scikit-learn. If `hdbscan` is installed, `run_section_clustering(...,
method="hdbscan")` can use it instead.


In [11]:
SECTION_EMBEDDING_COLUMNS = [
    "density_level_mean",
    "density_level_p90",
    "density_burst_p90",
    "hold_occupancy_mean",
    "hold_occupancy_p90",
    "ln_change_rate_mean",
    "chord_ratio_mean",
    "chord_ratio_p90",
    "jack_excess_mean",
    "jack_excess_p90",
    "jack_streak_exposure_p90",
    "hand_balance_signed_mean",
    "hand_imbalance_abs_mean",
    "repeat_exact_mean",
    "repeat_shift_mean",
    "repeat_motion_mean",
    "repeat_rhythm_mean",
]


def available_embedding_columns(section_df: pd.DataFrame) -> list[str]:
    return [column for column in SECTION_EMBEDDING_COLUMNS if column in section_df]


def filter_sections_for_clustering(
    section_df: pd.DataFrame,
    *,
    valid_fraction_min: float = 0.75,
    control_confidence_min: float = 0.3,
    edge_margin_s: float = 4.0,
) -> pd.DataFrame:
    df = section_df.copy()
    if df.empty:
        return df
    mask = pd.Series(True, index=df.index)
    if "valid_fraction" in df:
        mask &= pd.to_numeric(df["valid_fraction"], errors="coerce").fillna(0.0) >= valid_fraction_min
    if "control_confidence_mean" in df:
        mask &= pd.to_numeric(df["control_confidence_mean"], errors="coerce").fillna(0.0) >= control_confidence_min
    if "section_start_s" in df:
        mask &= pd.to_numeric(df["section_start_s"], errors="coerce").fillna(-math.inf) >= edge_margin_s
    if "section_end_s" in df and "map_duration_s" in df:
        section_end = pd.to_numeric(df["section_end_s"], errors="coerce")
        map_duration = pd.to_numeric(df["map_duration_s"], errors="coerce")
        mask &= section_end <= map_duration - edge_margin_s
    return df.loc[mask].reset_index(drop=True)


def robust_scaled_matrix(section_df: pd.DataFrame, columns: Sequence[str]) -> tuple[np.ndarray, dict[str, np.ndarray]]:
    values = section_df[list(columns)].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=float)
    median = np.median(values, axis=0)
    q25 = np.percentile(values, 25, axis=0)
    q75 = np.percentile(values, 75, axis=0)
    iqr = np.maximum(q75 - q25, 1e-6)
    scaled = (values - median) / iqr
    scaled = np.clip(scaled, -8.0, 8.0)
    return scaled, {"median": median, "iqr": iqr}


def pca2_numpy(x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    if len(x) == 0:
        return np.zeros((0, 2)), np.zeros((2, x.shape[1] if x.ndim == 2 else 0))
    centered = x - np.mean(x, axis=0, keepdims=True)
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    components = vt[:2]
    coords = centered @ components.T
    if coords.shape[1] == 1:
        coords = np.column_stack([coords[:, 0], np.zeros(len(coords))])
    return coords[:, :2], components


def kmeans_numpy(x: np.ndarray, *, n_clusters: int = 8, iterations: int = 100, random_state: int = 7) -> np.ndarray:
    if len(x) == 0:
        return np.zeros((0,), dtype=int)
    n_clusters = max(1, min(n_clusters, len(x)))
    rng = np.random.default_rng(random_state)
    centers = x[rng.choice(len(x), size=n_clusters, replace=False)].copy()
    labels = np.zeros(len(x), dtype=int)
    for _ in range(iterations):
        distances = np.sum((x[:, None, :] - centers[None, :, :]) ** 2, axis=2)
        new_labels = np.argmin(distances, axis=1)
        if np.array_equal(new_labels, labels):
            break
        labels = new_labels
        for k in range(n_clusters):
            members = x[labels == k]
            if len(members):
                centers[k] = np.mean(members, axis=0)
    return labels


def run_section_clustering(
    section_df: pd.DataFrame,
    *,
    columns: Sequence[str] | None = None,
    n_clusters: int = 8,
    method: str = "kmeans",
    random_state: int = 7,
    prefilter: bool = True,
    edge_margin_s: float = 4.0,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    input_sections = int(len(section_df))
    if prefilter:
        section_df = filter_sections_for_clustering(section_df, edge_margin_s=edge_margin_s)
    if section_df.empty:
        raise ValueError("no sections remain for clustering after filtering")
    if columns is None:
        columns = available_embedding_columns(section_df)
    if not columns:
        raise ValueError("no embedding columns are available")
    x, scale = robust_scaled_matrix(section_df, columns)
    pca_coords, components = pca2_numpy(x)

    if method == "hdbscan":
        try:
            import hdbscan  # type: ignore
        except ImportError as exc:
            raise ImportError("hdbscan is not installed; use method='kmeans' or install hdbscan") from exc
        labels = hdbscan.HDBSCAN(min_cluster_size=50, min_samples=10).fit_predict(x)
    elif method == "kmeans":
        labels = kmeans_numpy(x, n_clusters=n_clusters, random_state=random_state)
    else:
        raise ValueError("method must be 'kmeans' or 'hdbscan'")

    clustered = section_df.copy()
    clustered["cluster"] = labels
    clustered["pca1"] = pca_coords[:, 0]
    clustered["pca2"] = pca_coords[:, 1]
    diagnostics = {
        "columns": list(columns),
        "scale": scale,
        "pca_components": components,
        "method": method,
        "prefilter": prefilter,
        "input_sections": input_sections,
        "clustered_sections": int(len(clustered)),
    }
    return clustered, diagnostics


def cluster_profile(clustered: pd.DataFrame, *, columns: Sequence[str] | None = None) -> pd.DataFrame:
    if columns is None:
        columns = available_embedding_columns(clustered)
    summary = clustered.groupby("cluster")[list(columns)].mean(numeric_only=True)
    summary.insert(0, "sections", clustered.groupby("cluster").size())
    if "difficulty" in clustered:
        summary.insert(1, "difficulty_mean", clustered.groupby("cluster")["difficulty"].mean())
    if "map_duration_s" in clustered:
        summary.insert(2, "map_duration_mean", clustered.groupby("cluster")["map_duration_s"].mean())
    for extra in ["valid_fraction", "control_confidence_mean"]:
        if extra in clustered and f"{extra}_cluster_mean" not in summary:
            summary[f"{extra}_cluster_mean"] = clustered.groupby("cluster")[extra].mean()
    return summary.sort_values("sections", ascending=False)


def plot_cluster_scatter(clustered: pd.DataFrame, *, color_col: str = "cluster", alpha: float = 0.35):
    fig, ax = plt.subplots(1, 1, figsize=(9, 7))
    scatter = ax.scatter(clustered["pca1"], clustered["pca2"], c=clustered[color_col], s=10, alpha=alpha, cmap="tab10")
    ax.set_xlabel("robust PCA 1")
    ax.set_ylabel("robust PCA 2")
    ax.set_title(f"Section embedding clusters colored by {color_col}")
    ax.grid(True, alpha=0.2)
    fig.colorbar(scatter, ax=ax, label=color_col)
    fig.tight_layout()
    return fig


def cluster_axis_dominance(clustered: pd.DataFrame, *, columns: Sequence[str] | None = None) -> pd.DataFrame:
    if columns is None:
        columns = available_embedding_columns(clustered)
    rows = []
    labels = clustered["cluster"].to_numpy()
    for column in columns:
        values = pd.to_numeric(clustered[column], errors="coerce").to_numpy(dtype=float)
        finite = np.isfinite(values)
        if finite.sum() < 3:
            eta2 = math.nan
        else:
            total_mean = float(np.mean(values[finite]))
            ss_total = float(np.sum((values[finite] - total_mean) ** 2))
            ss_between = 0.0
            for label in np.unique(labels[finite]):
                group = values[finite & (labels == label)]
                if len(group):
                    ss_between += len(group) * (float(np.mean(group)) - total_mean) ** 2
            eta2 = ss_between / ss_total if ss_total > 1e-12 else 0.0
        rows.append({"column": column, "cluster_eta2": eta2})
    return pd.DataFrame(rows).sort_values("cluster_eta2", ascending=False).reset_index(drop=True)


def density_bucketed_section_clustering(
    section_df: pd.DataFrame,
    *,
    density_col: str = "density_level_mean",
    q: int = 3,
    n_clusters: int = 4,
    min_sections: int = 30,
    random_state: int = 7,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    filtered = filter_sections_for_clustering(section_df)
    if density_col not in filtered:
        raise KeyError(density_col)
    filtered = filtered.copy()
    filtered["density_bucket"] = pd.qcut(
        pd.to_numeric(filtered[density_col], errors="coerce").rank(method="first"),
        q=q,
        labels=[f"density_{i}" for i in range(q)],
        duplicates="drop",
    )
    clustered_parts = []
    diagnostics: dict[str, Any] = {"buckets": {}, "input_sections": int(len(section_df)), "filtered_sections": int(len(filtered))}
    for bucket, group in filtered.groupby("density_bucket", observed=True):
        if len(group) < min_sections:
            diagnostics["buckets"][str(bucket)] = {"sections": int(len(group)), "skipped": True}
            continue
        clustered, info = run_section_clustering(
            group.drop(columns=["cluster", "pca1", "pca2"], errors="ignore"),
            n_clusters=min(n_clusters, len(group)),
            random_state=random_state,
            prefilter=False,
        )
        clustered["density_bucket"] = str(bucket)
        clustered["density_bucket_cluster"] = clustered["density_bucket"] + ":" + clustered["cluster"].astype(str)
        clustered_parts.append(clustered)
        diagnostics["buckets"][str(bucket)] = {"sections": int(len(group)), "skipped": False, "info": info}
    if not clustered_parts:
        return pd.DataFrame(), diagnostics
    return pd.concat(clustered_parts, ignore_index=True), diagnostics


def cluster_representative_sections(
    clustered: pd.DataFrame,
    *,
    columns: Sequence[str] | None = None,
    per_cluster: int = 5,
    strategy: str = "medoid",
) -> pd.DataFrame:
    if columns is None:
        columns = available_embedding_columns(clustered)
    if "cluster" not in clustered:
        raise KeyError("cluster")
    rows = []
    for cluster, group in clustered.groupby("cluster"):
        numeric = group[list(columns)].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=float)
        if len(group) == 0:
            continue
        if strategy == "medoid":
            center = np.mean(numeric, axis=0, keepdims=True)
            order = np.argsort(np.sum((numeric - center) ** 2, axis=1))
        elif strategy == "high_confidence":
            order = np.argsort(-_section_numeric(group, "control_confidence_mean").to_numpy(dtype=float))
        elif strategy == "top_axis":
            axis = cluster_axis_dominance(group, columns=columns).head(1)["column"].iloc[0]
            order = np.argsort(-_section_numeric(group, axis).to_numpy(dtype=float))
        else:
            raise ValueError("strategy must be 'medoid', 'high_confidence', or 'top_axis'")
        selected = group.iloc[order[:per_cluster]].copy()
        selected["representative_strategy"] = strategy
        rows.append(selected)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def cluster_stability_report(
    section_df: pd.DataFrame,
    *,
    seeds: Sequence[int] = (1, 3, 5, 7, 11),
    n_clusters: int = 8,
    columns: Sequence[str] | None = None,
) -> pd.DataFrame:
    rows = []
    for seed in seeds:
        clustered, info = run_section_clustering(section_df, columns=columns, n_clusters=n_clusters, random_state=seed)
        dominance = cluster_axis_dominance(clustered, columns=info["columns"]).head(5)
        rows.append({
            "seed": int(seed),
            "clustered_sections": int(len(clustered)),
            "top_axes": ",".join(dominance["column"].astype(str).tolist()),
            "top_axis_eta2_mean": float(dominance["cluster_eta2"].mean()) if not dominance.empty else math.nan,
        })
    return pd.DataFrame(rows)


## Example Runs

The first two examples are cheap and safe to run. The dataset pass is intentionally opt-in because it
recomputes V2 features from source `.osu` files. Start with `sample_size=256`; set `sample_size=None`
only when you are ready for the full eligible corpus.


In [12]:
RUN_SYNTHETIC_EXAMPLE = True
RUN_PERTURBATION_EXAMPLE = True
RUN_DATASET_AUDIT = True
RUN_DATASET_AUDIT_FROM_TIMESERIES = True
RUN_CLUSTERING_EXAMPLE = False

if RUN_SYNTHETIC_EXAMPLE:
    synthetic_outputs, synthetic_summary = synthetic_probe_outputs(duration_s=16.0)
    display(synthetic_summary[[
        "fixture",
        "objects",
        "density_level_p95",
        "density_burst_p95",
        "hold_occupancy_p95",
        "ln_change_rate_p95",
        "chord_ratio_p95",
        "jack_excess_p95",
        "jack_streak_exposure_p95",
        "hand_imbalance_abs_p95",
        "repeat_exact_p95",
        "repeat_shift_p95",
        "repeat_motion_p95",
        "repeat_rhythm_p95",
    ]])
    display(evaluate_synthetic_expectations(synthetic_summary))

if RUN_PERTURBATION_EXAMPLE:
    stream_hits = fixture_single_stream(16.0)
    random_hits = fixture_random_stream(16.0)
    normal_roll_hits = fixture_normal_roll(16.0)
    normal_jack_hits = fixture_normal_jack(16.0)
    double_stair_hits = fixture_double_stair(16.0)
    mixed_hits = fixture_mixed_ln_chord_section(16.0)
    perturbation_examples = {
        "single_stream_mirror": mirror_invariance_report(stream_hits, duration_s=16.0),
        "single_stream_time_stretch_1p1": time_stretch_delta_table(stream_hits, duration_s=16.0, factor=1.1),
        "normal_jack_time_stretch_1p1": time_stretch_delta_table(normal_jack_hits, duration_s=16.0, factor=1.1),
        "random_stream_column_shuffle": perturbation_delta_table(random_hits, column_shuffle_preserving_chord_size(random_hits, seed=99), duration_s=16.0),
        "normal_jack_column_shuffle": perturbation_delta_table(normal_jack_hits, column_shuffle_preserving_chord_size(normal_jack_hits, seed=99), duration_s=16.0),
        "double_stair_event_order_randomize": perturbation_delta_table(double_stair_hits, randomize_event_order_within_section(double_stair_hits), duration_s=16.0),
        "single_stream_event_order_randomize": perturbation_delta_table(stream_hits, randomize_event_order_within_section(stream_hits), duration_s=16.0),
        "single_stream_singles_to_doubles": perturbation_delta_table(stream_hits, convert_singles_to_doubles(stream_hits), duration_s=16.0),
        "normal_roll_force_same_column": perturbation_delta_table(normal_roll_hits, force_same_column_repeats(normal_roll_hits), duration_s=16.0),
        "mixed_remove_ln_tails": perturbation_delta_table(mixed_hits, remove_ln_tails(mixed_hits), duration_s=16.0, summary_stat="mean"),
        "mixed_extend_ln_tails": perturbation_delta_table(mixed_hits, extend_ln_tails_only(mixed_hits), duration_s=17.5, summary_stat="mean"),
    }
    for name, table in perturbation_examples.items():
        print("\n", name)
        display(table.head(12))
    display(evaluate_perturbation_expectations(perturbation_examples))

if RUN_DATASET_AUDIT:
    if RUN_DATASET_AUDIT_FROM_TIMESERIES:
        section_df, error_df = build_section_audit_from_timeseries_parquet(sample_size=None, seed=7, section_s=8.0, stride_s=4.0)
    else:
        section_df, error_df = build_dataset_section_audit(sample_size=256, seed=7, section_s=8.0, stride_s=4.0)
    display(error_df.head(20))
    display(distribution_report(section_df, stat="mean"))
    display(saturation_report(section_df))
    display(low_confidence_high_value_report(section_df, n=25))
    for method, corr in feature_correlation_reports(section_df).items():
        print(method)
        display(corr)
    display(selected_partial_correlations(section_df))
    display(evaluate_control_v2_audit(section_df))

if RUN_CLUSTERING_EXAMPLE:
    if RUN_DATASET_AUDIT_FROM_TIMESERIES:
        section_df = load_or_build_section_audit_from_timeseries(sample_size=None, seed=7)
    else:
        section_df = load_or_build_section_audit(sample_size=256, seed=7)
    section_df = add_analysis_buckets(section_df)
    cluster_input = filter_sections_for_clustering(section_df)
    print(f"clustering sections: {len(cluster_input)}/{len(section_df)} after validity/confidence/edge filtering")
    clustered, clustering_info = run_section_clustering(cluster_input, n_clusters=8, method="kmeans", prefilter=False)
    display(cluster_profile(clustered))
    display(cluster_axis_dominance(clustered).head(12))
    display(cluster_representative_sections(clustered, per_cluster=3).head(24))
    display(cluster_stability_report(cluster_input, n_clusters=8))
    plot_cluster_scatter(clustered)


,fixture,objects,density_level_p95,density_burst_p95,hold_occupancy_p95,ln_change_rate_p95,chord_ratio_p95,jack_excess_p95,jack_streak_exposure_p95,hand_imbalance_abs_p95,repeat_exact_p95,repeat_shift_p95,repeat_motion_p95,repeat_rhythm_p95
0,rest,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,single_stream,65,1.609438,0.781806,0.000000,0.000000,0.000000,0.000000,0.000000,0.036765,0.204364,0.718681,0.204364,0.976644
2,random_stream,129,2.197225,0.966124,0.000000,0.000000,0.000000,0.426440,0.184112,0.174242,0.189405,0.331457,0.207455,0.999740
3,chordstream,130,2.290164,0.981248,0.000000,0.000000,0.298562,0.000000,0.000000,0.036675,0.204364,0.204364,0.204364,0.976644
4,jumpstream,133,2.297814,0.983009,0.000000,0.000000,0.162170,0.000000,0.000000,0.083350,0.137466,0.137466,0.137466,0.995915
5,normal_jack,81,1.791759,0.867336,0.000000,0.000000,0.000000,0.616803,0.031053,0.625000,0.992413,0.992413,0.992413,0.992413
6,stress_jack,161,2.397895,0.989474,0.000000,0.000000,0.000000,0.904048,0.955505,0.769231,0.999972,0.999972,0.999972,0.999972
7,minijack,134,2.233592,0.975384,0.000000,0.000000,0.000000,0.422441,0.139720,0.047463,0.106485,0.488644,0.242864,0.999821
8,anchor,201,2.668700,0.998076,0.000000,0.000000,0.166812,0.419193,0.870578,0.448905,0.149232,0.149232,0.149232,0.999821
9,single_stair,54,1.466337,0.685754,0.000000,0.000000,0.000000,0.000000,0.000000,0.055800,0.196367,0.705089,0.196367,0.950524


,fixture,feature,statistic,value,op,threshold,pass,reason
0,rest,density_level,max,0.000000,<=,0.00,True,rest should stay neutral
1,rest,repeat_exact,max,0.000000,<=,0.00,True,no objects means no recurrence
2,normal_jack,jack_excess,p95,0.616803,>=,0.30,True,ranked-normal same-column gaps should exceed null
3,normal_jack,repeat_exact,p95,0.992413,>=,0.50,True,literal recurrence should be high on normal jack
4,single_stair,repeat_motion,p95,0.196367,>=,0.10,True,single-column movement recurrence should fire
5,single_stair,jack_excess,p95,0.000000,<=,0.05,True,single stair should not look like jack risk
6,double_stair,repeat_shift,p95,0.524166,>=,0.45,True,shift-normalized chord-shape recurrence should...
7,double_stair,jack_excess,p95,0.000000,<=,0.05,True,double stair should not look like jack risk
8,jumpstream,density_level,p95,2.297814,>=,1.80,True,single-chord jumpstream should be high-density
9,jumpstream,chord_ratio,p95,0.162170,>=,0.15,True,jumpstream alternates singles and chords



 single_stream_mirror


,feature,invariance_error,abs_delta,expected
0,density_level,0.0,0.0,NaN
1,density_burst,0.0,0.0,NaN
2,hold_occupancy,0.0,0.0,NaN
3,ln_change_rate,0.0,0.0,NaN
4,chord_ratio,0.0,0.0,NaN
5,jack_excess,0.0,0.0,NaN
6,jack_streak_exposure,0.0,0.0,NaN
7,repeat_exact,0.0,0.0,NaN
8,repeat_shift,0.0,0.0,NaN
9,repeat_motion,0.0,0.0,NaN



 single_stream_time_stretch_1p1


...

,density_level_mean,density_burst_mean,hold_occupancy_mean,ln_change_rate_mean,chord_ratio_mean,jack_excess_mean,jack_streak_exposure_mean,hand_balance_signed_mean,hand_imbalance_abs_mean,repeat_exact_mean,...,repeat_motion_mean,repeat_rhythm_mean,density_confidence_mean,ln_change_confidence_mean,chord_confidence_mean,jack_confidence_mean,jack_streak_confidence_mean,hand_confidence_mean,repeat_confidence_mean,control_confidence_mean
density_level_mean,1.000000,0.149139,-0.200326,-0.015605,0.534268,-0.016197,0.612830,-0.006401,-0.306824,-0.052387,...,-0.080877,0.393975,0.437292,-0.029283,0.756683,0.806972,0.779433,0.976271,0.621435,0.736082
density_burst_mean,0.149139,1.000000,-0.121888,-0.080775,0.060713,-0.017882,0.130019,-0.000055,-0.075400,0.022227,...,0.023775,0.081714,0.083549,-0.083751,0.136178,0.141124,0.132655,0.148511,0.103215,0.085763
hold_occupancy_mean,-0.200326,-0.121888,1.000000,0.837371,0.003597,-0.031876,-0.203471,-0.002013,0.120210,-0.124048,...,-0.138508,-0.141395,-0.108779,0.756971,-0.192722,-0.194644,-0.170472,-0.198370,-0.157231,0.207600
ln_change_rate_mean,-0.015605,-0.080775,0.837371,1.000000,0.076536,-0.039042,-0.181904,0.000857,0.039125,-0.121269,...,-0.137221,-0.091237,0.065962,0.971460,0.053005,-0.026639,-0.103574,0.015682,0.065852,0.510740
chord_ratio_mean,0.534268,0.060713,0.003597,0.076536,1.000000,-0.017245,0.233605,-0.003066,-0.162620,-0.158956,...,-0.193098,0.198238,0.160451,0.069857,0.263464,0.164549,0.347266,0.480582,0.238987,0.271807
jack_excess_mean,-0.016197,-0.017882,-0.031876,-0.039042,-0.017245,1.000000,0.057776,0.017450,0.214896,0.205392,...,0.240178,0.051495,0.012539,-0.039429,0.021558,0.017709,0.074915,-0.006858,0.020022,-0.005821
jack_streak_exposure_mean,0.612830,0.130019,-0.203471,-0.181904,0.233605,0.057776,1.000000,-0.008293,-0.160055,-0.016148,...,-0.017676,0.202068,0.059868,-0.207538,0.213759,0.429986,0.780888,0.488089,0.123639,0.212438
hand_balance_signed_mean,-0.006401,-0.000055,-0.002013,0.000857,-0.003066,0.017450,-0.008293,1.000000,0.036798,-0.013494,...,-0.011830,-0.010655,0.001019,0.002126,-0.001426,-0.005308,-0.005675,-0.005027,-0.000079,-0.002137
hand_imbalance_abs_mean,-0.306824,-0.075400,0.120210,0.039125,-0.162620,0.214896,-0.160055,0.036798,1.000000,0.332579,...,0.274296,-0.167038,-0.023003,0.037170,-0.195051,-0.271200,-0.205488,-0.284285,-0.124078,-0.194629
repeat_exact_mean,-0.052387,0.022227,-0.124048,-0.121269,-0.158956,0.205392,-0.016148,-0.013494,0.332579,1.000000,...,0.889187,0.238700,0.152635,-0.117204,0.095178,-0.005444,-0.061574,-0.014395,0.142947,-0.015072


spearman


,density_level_mean,density_burst_mean,hold_occupancy_mean,ln_change_rate_mean,chord_ratio_mean,jack_excess_mean,jack_streak_exposure_mean,hand_balance_signed_mean,hand_imbalance_abs_mean,repeat_exact_mean,...,repeat_motion_mean,repeat_rhythm_mean,density_confidence_mean,ln_change_confidence_mean,chord_confidence_mean,jack_confidence_mean,jack_streak_confidence_mean,hand_confidence_mean,repeat_confidence_mean,control_confidence_mean
density_level_mean,1.000000,0.134344,-0.240286,-0.112329,0.526930,-0.005490,0.856714,-0.002896,-0.375796,-0.082334,...,-0.110150,0.328497,0.852167,-0.095384,0.867654,0.842678,0.875995,0.997502,0.869683,0.613242
density_burst_mean,0.134344,1.000000,-0.105519,-0.075288,0.047676,-0.026804,0.071580,0.000521,-0.088296,0.034002,...,0.032363,0.076505,0.085875,-0.071081,0.127993,0.143891,0.098162,0.136442,0.103241,0.053769
hold_occupancy_mean,-0.240286,-0.105519,1.000000,0.929427,0.034543,0.002790,-0.202014,0.005515,0.199806,-0.143286,...,-0.160199,-0.165341,-0.268183,0.887332,-0.287232,-0.275309,-0.213347,-0.251240,-0.280830,0.381612
ln_change_rate_mean,-0.112329,-0.075288,0.929427,1.000000,0.094932,-0.002874,-0.126713,0.006388,0.131253,-0.132386,...,-0.151959,-0.118681,-0.124426,0.988417,-0.144783,-0.148243,-0.132558,-0.119350,-0.140061,0.572190
chord_ratio_mean,0.526930,0.047676,0.034543,0.094932,1.000000,0.118298,0.386966,0.000367,-0.176145,-0.232841,...,-0.276097,0.181156,0.149180,0.098182,0.125077,0.105103,0.380494,0.480031,0.130178,0.257853
jack_excess_mean,-0.005490,-0.026804,0.002790,-0.002874,0.118298,1.000000,0.158230,-0.004604,0.140043,-0.001961,...,0.048156,0.014567,-0.070453,-0.005012,-0.075880,-0.077063,0.118963,-0.015268,-0.073202,-0.030259
jack_streak_exposure_mean,0.856714,0.071580,-0.202014,-0.126713,0.386966,0.158230,1.000000,-0.003564,-0.244312,-0.078603,...,-0.076764,0.206900,0.693757,-0.123050,0.720022,0.744011,0.970485,0.848626,0.724478,0.488858
hand_balance_signed_mean,-0.002896,0.000521,0.005515,0.006388,0.000367,-0.004604,-0.003564,1.000000,-0.003596,-0.005427,...,-0.006678,-0.007834,-0.001471,0.006917,-0.002398,-0.002273,-0.002867,-0.002950,-0.002130,0.004839
hand_imbalance_abs_mean,-0.375796,-0.088296,0.199806,0.131253,-0.176145,0.140043,-0.244312,-0.003596,1.000000,0.058336,...,0.058989,-0.215200,-0.330196,0.116463,-0.340783,-0.333562,-0.269179,-0.376755,-0.339539,-0.182400
repeat_exact_mean,-0.082334,0.034002,-0.143286,-0.132386,-0.232841,-0.001961,-0.078603,-0.005427,0.058336,1.000000,...,0.877907,0.292355,0.018175,-0.127572,0.028231,0.035272,-0.083444,-0.069034,0.024886,-0.077076


,x,y,pearson,spearman,controls,partial
0,jack_excess_mean,density_level_mean,-0.016197,-0.005490,chord_ratio_mean,-0.008263
1,jack_excess_mean,density_level_mean,-0.016197,-0.005490,density_raw_med_mean,0.025336
2,repeat_motion_mean,density_level_mean,-0.080877,-0.110150,chord_ratio_mean,0.026874
3,repeat_exact_mean,jack_excess_mean,0.205392,-0.001961,density_level_mean,0.204852
4,ln_change_rate_mean,hold_occupancy_mean,0.837371,0.929427,density_level_mean,0.851610
5,chord_ratio_mean,density_level_mean,0.534268,0.526930,,NaN
6,repeat_shift_mean,repeat_motion_mean,0.737092,0.708120,density_level_mean,0.735487
7,hand_imbalance_abs_mean,density_level_mean,-0.306824,-0.375796,,NaN


,gate_name,metric,value,threshold,pass,reason
0,finite_model_stats,finite_rate,1.000000,>= 0.999,True,model section statistics should be finite
1,valid_fraction,mean,0.993986,>= 0.80,True,section windows should mostly cover valid map ...
2,low_confidence_high_value,sections_per_input_section,0.564628,<= 0.10,False,high value channels should not usually occur a...
3,chord_ratio_near_high,near_high_rate,0.000010,<= 0.05,True,bounded channels should not saturate high acro...
4,repeat_exact_near_high,near_high_rate,0.000037,<= 0.05,True,bounded channels should not saturate high acro...
5,repeat_exact_near_zero,near_zero_rate,0.002834,<= 0.98,True,repeat channels should not be structurally zero
6,repeat_shift_near_high,near_high_rate,0.000044,<= 0.05,True,bounded channels should not saturate high acro...
7,repeat_shift_near_zero,near_zero_rate,0.002790,<= 0.98,True,repeat channels should not be structurally zero
8,repeat_motion_near_high,near_high_rate,0.000037,<= 0.05,True,bounded channels should not saturate high acro...
9,repeat_motion_near_zero,near_zero_rate,0.002805,<= 0.98,True,repeat channels should not be structurally zero
